# 包

In [ ]:
pip install Bio

In [ ]:
pip install torch torch-geometric optuna tqdm scikit-learn esm umap-learn shap xgboost

In [ ]:
!pip uninstall -y numpy pandas
!pip install numpy==1.26.4 pandas==2.2.2

In [ ]:
pip list

In [ ]:
!pip show numpy

In [ ]:
!pip show pandas

In [ ]:
!pip show thinc

In [ ]:
!python --version

Python 3.12.11


In [ ]:
pip install biopython pandas matplotlib Bio esm

# 数据集

## 阳性样本cd-hit

In [ ]:
!apt-get install cd-hit

In [ ]:
!grep -c "^>" AFP.fasta

In [ ]:
!cd-hit -i AFP.fasta -o AFP_clustered.fasta -c 0.8 -n 5

In [ ]:
def convert_to_fasta(input_filename, output_filename):
    with open(input_filename, 'r') as file:
        sequences = file.readlines()

    with open(output_filename, 'w') as file:
        for i, sequence in enumerate(sequences):
            sequence = sequence.strip()  
            file.write(f">Sequence_{i + 1}\n{sequence}\n")
convert_to_fasta('AFP_2.txt', 'AFP_2.fasta')


In [ ]:
with open("AFP.fasta", "r") as file:
    sequences = file.read().split('>')
    sequences = [seq for seq in sequences if seq.strip()]
    unique_sequences = set(sequences)

print(f"Total sequences: {len(sequences)}")
print(f"Unique sequences: {len(unique_sequences)}")


In [ ]:
def filter_sequences(input_file, output_file, max_length=100):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        write_sequence = False
        for line in infile:
            if line.startswith('>'):
                if write_sequence:
                    outfile.write(sequence_header + sequence_data)
                sequence_header = line
                sequence_data = ''
                write_sequence = False  # Reset for next sequence
            else:
                sequence_data += line
                if len(sequence_data.replace('\n', '')) <= max_length:
                    write_sequence = True
                else:
                    write_sequence = False

        # Check last sequence
        if write_sequence:
            outfile.write(sequence_header + sequence_data)

input_filename = 'AFP_clustered.fasta'
output_filename = 'AFP_CD-hit.fasta'
filter_sequences(input_filename, output_filename)


In [ ]:
def renumber_sequences(input_file, output_file):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        counter = 1
        for line in infile:
            if line.startswith('>'):
                outfile.write(f'>Sequence_{counter}\n')
                counter += 1
            else:
                outfile.write(line)

input_filename = 'AFP_CD-hit.fasta' 
output_filename = 'AFP_renumbered.fasta'
renumber_sequences(input_filename, output_filename)


## 阴性样本cd-hit

In [ ]:
!apt-get install cd-hit

In [ ]:
def convert_to_fasta(input_filename, output_filename):

    with open(input_filename, 'r') as file:
        sequences = file.readlines()

    with open(output_filename, 'w') as file:
        for i, sequence in enumerate(sequences):
            sequence = sequence.strip()  
            file.write(f">Sequence_{i + 1}\n{sequence}\n")

convert_to_fasta('Non_AFP.txt', 'Non_AFP.fasta')


In [ ]:
!cd-hit -i Non_AFP.fasta -o Non_AFP_clustered.fasta -c 0.8 -n 5

In [ ]:
def filter_sequences(input_file, output_file, max_length=100):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        write_sequence = False
        for line in infile:
            if line.startswith('>'):
                if write_sequence:
                    outfile.write(sequence_header + sequence_data)
                sequence_header = line
                sequence_data = ''
                write_sequence = False  # Reset for next sequence
            else:
                sequence_data += line
                if len(sequence_data.replace('\n', '')) <= max_length:
                    write_sequence = True
                else:
                    write_sequence = False

        # Check last sequence
        if write_sequence:
            outfile.write(sequence_header + sequence_data)

input_filename = 'Non_AFP_clustered.fasta'
output_filename = 'Non_AFP_CD-hit.fasta'
filter_sequences(input_filename, output_filename)


In [ ]:
def renumber_sequences(input_file, output_file):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        counter = 1
        for line in infile:
            if line.startswith('>'):
                outfile.write(f'>Sequence_{counter}\n')
                counter += 1
            else:
                outfile.write(line)

input_filename = 'Non_AFP_CD-hit.fasta'  
output_filename = 'Non_AFP_renumbered.fasta'
renumber_sequences(input_filename, output_filename)


## 划分数据集


In [ ]:
pip install Bio

In [ ]:
from Bio import SeqIO
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split

def load_sequences(file_path, label):
    sequences = []
    labels = []
    for record in SeqIO.parse(file_path, "fasta"):
        sequences.append(str(record.seq))
        labels.append(label)
    return sequences, labels

def balance_and_split(sequences, labels):
    df = pd.DataFrame({
        'sequence': sequences,
        'label': labels
    })

    print(f"原始数据总量: {df.shape[0]}")
    print(f"各类样本数量：\n{df['label'].value_counts()}")

    # 进行欠采样
    rus = RandomUnderSampler(random_state=42)
    X_res, y_res = rus.fit_resample(df[['sequence']], df['label'])

    print(f"欠采样后数据总量: {X_res.shape[0]}")
    print(f"欠采样后各类样本数量：\n{pd.Series(y_res).value_counts()}")

    # 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)

    print(f"训练集数量: {X_train.shape[0]}")
    print(f"测试集数量: {X_test.shape[0]}")

    train_df = pd.DataFrame(X_train, columns=['sequence'])
    train_df['label'] = y_train
    test_df = pd.DataFrame(X_test, columns=['sequence'])
    test_df['label'] = y_test

    train_df.to_csv('train_dataset.csv', index=False)
    test_df.to_csv('test_dataset.csv', index=False)

    return train_df, test_df

# 加载阳性和阴性样本，标签分别设置为 1 和 0
pos_sequences, pos_labels = load_sequences('AFP_renumbered.fasta', 1)
neg_sequences, neg_labels = load_sequences('Non_AFP_renumbered.fasta', 0)

print(f"阳性样本数量: {len(pos_sequences)}")
print(f"阴性样本数量: {len(neg_sequences)}")

all_sequences = pos_sequences + neg_sequences
all_labels = pos_labels + neg_labels

train_df, test_df = balance_and_split(all_sequences, all_labels)


## 去除非20个标准氨基酸

In [ ]:
import pandas as pd

def replace_non_standard_amino_acids(seq):
    replacements = {'B': 'D', 'Z': 'E', 'X': 'A', 'J': 'L', 'U': 'C', 'O': 'K'}
    for old, new in replacements.items():
        seq = seq.replace(old, new)
    return seq

def load_and_preprocess(file_path, output_path):
    df = pd.read_csv(file_path)
    df['sequence'] = df['sequence'].apply(replace_non_standard_amino_acids)
    df.to_csv(output_path, index=False)
    return df

train_df = load_and_preprocess('train_dataset.csv', 'processed_train_dataset.csv')
test_df = load_and_preprocess('test_dataset.csv', 'processed_test_dataset.csv')

print("Processed datasets have been saved.")


In [ ]:
def renumber_sequences(input_file, output_file):
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        counter = 1
        for line in infile:
            if line.startswith('>'):
                outfile.write(f'>Sequence_{counter}\n')
                counter += 1
            else:
                outfile.write(line)

input_filename = 'processed_test_dataset.csv'
output_filename = 'ColabFold_test_dataset.csv'
renumber_sequences(input_filename, output_filename)


## 分别划分成训练集阳性，训练集阴性，测试集阳性，测试集阴性

In [ ]:
import pandas as pd

def to_fasta(df, file_name):
    with open(file_name, 'w') as f:
        for index, row in df.iterrows():
            f.write(f">{index}\n{row['sequence']}\n")

def process_and_save(data_path, output_prefix):
    df = pd.read_csv(data_path)
    pos = df[df['label'] == 1]
    neg = df[df['label'] == 0]
    to_fasta(pos, f'{output_prefix}_pos.fasta')
    to_fasta(neg, f'{output_prefix}_neg.fasta')

train_path = 'processed_train_dataset.csv'
test_path = 'processed_test_dataset.csv'

# 处理训练集和测试集，保存为FASTA文件
# /content/drive/MyDrive/AFP_work/seq/Colab_train_neg.fasta
process_and_save(train_path, 'Colab_train')
process_and_save(test_path, 'Colab_test')

# 结合

## ESM-C

In [ ]:
import os
from Bio import SeqIO
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
import torch
import numpy as np
from tqdm import tqdm

# 定义 FASTA 文件路径
#train_pos_seq_file = '/content/drive/MyDrive/AFP_work/seq/Colab_train_pos.fasta'
#train_neg_seq_file = '/content/drive/MyDrive/AFP_work/seq/Colab_train_neg.fasta'
test_pos_seq_file = '/content/drive/MyDrive/AFP_work/seq/Colab_test_pos.fasta'
#test_neg_seq_file = '/content/drive/MyDrive/AFP_work/seq/Colab_test_neg.fasta'

output_feature_path = '/content/drive/MyDrive/esmc_600_test_pos'
os.makedirs(output_feature_path, exist_ok=True)

def read_fasta_sequences(fasta_file):
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq_id = record.id
        sequence = str(record.seq).replace(" ", "").replace("\n", "")
        sequences.append((seq_id, sequence))
    return sequences

def save_features(features, output_dir):
      for seq_id, feature_dict in features.items():
        logits = feature_dict['logits']
        embeddings = feature_dict['embeddings']
        logits_path = os.path.join(output_dir, f"{seq_id}_logits.npy")
        embeddings_path = os.path.join(output_dir, f"{seq_id}_embeddings.npy")
        np.save(logits_path, logits)
        np.save(embeddings_path, embeddings)

def extract_features_individual(client, sequences):
    features = {}
    for seq_id, seq in tqdm(sequences, desc="提取特征"):
        try:
            protein = ESMProtein(sequence=seq)
            protein_tensor = client.encode(protein)
            logits_output = client.logits(
                protein_tensor,
                LogitsConfig(sequence=True, return_embeddings=True)
            )

            logits = logits_output.logits 
            embeddings = logits_output.embeddings

            if isinstance(logits, torch.Tensor):
                logits = logits.cpu().numpy()
            elif isinstance(logits, np.ndarray):
                pass
            else:
                logits = np.array(logits)

            if isinstance(embeddings, torch.Tensor):
                embeddings = embeddings.cpu().numpy()
            elif isinstance(embeddings, np.ndarray):
                pass
            else:
                embeddings = np.array(embeddings)

            features[seq_id] = {
                'logits': logits,
                'embeddings': embeddings
            }
        except Exception as e:
            print(f"处理序列 {seq_id} 时出错: {e}")
            continue
    return features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
client = ESMC.from_pretrained("esmc_600m").to(device)
client.eval()

fasta_files = [
    #train_pos_seq_file,
    #train_neg_seq_file,
    test_pos_seq_file,
    #test_neg_seq_file
]

for fasta_file in fasta_files:
    print(f"正在处理文件: {fasta_file}")
    sequences = read_fasta_sequences(fasta_file)
    print(f"序列数量: {len(sequences)}")
    features = extract_features_individual(client, sequences)
    save_features(features, output_feature_path)
    print(f"特征已保存: {fasta_file}\n")


In [ ]:
import os
import glob
from collections import defaultdict

# feature_dirs = [
#     '/content/drive/MyDrive/AFP_work/esmc_600_train_pos',
#     '/content/drive/MyDrive/AFP_work/esmc_600_train_neg',
#     '/content/drive/MyDrive/AFP_work/esmc_600_test_pos',
#     '/content/drive/MyDrive/AFP_work/esmc_600_test_neg'
# ]

# feature_dirs = [
#     'external_set_result/esmc_600_external_test/esmc_600_external_pos',
#     'external_set_result/esmc_600_external_test/esmc_600_external_neg'
# ]


feature_dirs = [
    'external_set_result/esmc_600_external_test/esmc_600_external_pos',
    'external_set_result/esmc_600_external_test/esmc_600_external_neg'
]

def count_and_verify_files(feature_dirs):
    for feature_dir in feature_dirs:
        print(f"正在处理文件夹: {feature_dir}")
        if not os.path.isdir(feature_dir):
            print(f"文件夹 '{feature_dir}' 不存在，跳过。\n")
            continue
        logits_files = sorted(glob.glob(os.path.join(feature_dir, '*_logits.npy')))
        embeddings_files = sorted(glob.glob(os.path.join(feature_dir, '*_embeddings.npy')))

        num_logits = len(logits_files)
        num_embeddings = len(embeddings_files)

        print(f"找到 {num_logits} 个 logits 文件和 {num_embeddings} 个 embeddings 文件。")
        if num_logits == num_embeddings:
            print("logits 文件和 embeddings 文件数量一致。\n")
        else:
            print("logits 文件和 embeddings 文件数量不一致。")
            print(f"logits 文件数量: {num_logits}, embeddings 文件数量: {num_embeddings}")

            logits_indices = set([os.path.basename(f).split('_')[0] for f in logits_files])
            embeddings_indices = set([os.path.basename(f).split('_')[0] for f in embeddings_files])

            missing_in_embeddings = logits_indices - embeddings_indices
            missing_in_logits = embeddings_indices - logits_indices

            if missing_in_embeddings:
                print(f"在 embeddings 文件夹中缺失以下 indices 的文件: {sorted(missing_in_embeddings)}")
            if missing_in_logits:
                print(f"在 logits 文件夹中缺失以下 indices 的文件: {sorted(missing_in_logits)}")
            print("\n")

count_and_verify_files(feature_dirs)


In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from sklearn.decomposition import PCA

def combine_features_with_pooling(feature_dir, pool_method='average', embedding_dim=None):
     # 查找所有 logits 和 embeddings 文件，排除以 'combined_' 开头的文件
    logits_files = sorted(
        [f for f in glob(os.path.join(feature_dir, '*_logits.npy'))
         if not os.path.basename(f).startswith('combined_')],
        # key=lambda x: int(os.path.basename(x).split('_')[0])
        key=lambda x: int(os.path.basename(x).split('_')[1])
    )
    embeddings_files = sorted(
        [f for f in glob(os.path.join(feature_dir, '*_embeddings.npy'))
         if not os.path.basename(f).startswith('combined_')],
        # key=lambda x: int(os.path.basename(x).split('_')[0])
        key=lambda x: int(os.path.basename(x).split('_')[1])
    )

    print(f"正在处理文件夹: {feature_dir}")
    print(f"找到 {len(logits_files)} 个 logits 文件和 {len(embeddings_files)} 个 embeddings 文件。")

    if len(logits_files) != len(embeddings_files):
        print("logits 文件和 embeddings 文件数量不一致，跳过此文件夹。\n")
        return

    logits_list = []
    embeddings_list = []
    seq_indices = []

    for logits_file, embeddings_file in tqdm(zip(logits_files, embeddings_files),
                                            total=len(logits_files), desc="加载文件"):
        try:
            logits = np.load(logits_file, allow_pickle=True)
            embeddings = np.load(embeddings_file, allow_pickle=True)
            if logits.ndim == 1:
                logits = logits.reshape(1, -1)
            logits_list.append(logits)

            if embeddings.ndim == 3:
                embeddings = embeddings.squeeze(0)  # 去除 batch_size 维度
            if pool_method == 'average':
                pooled_embeddings = embeddings.mean(axis=0)  # 平均池化
            elif pool_method == 'max':
                pooled_embeddings = embeddings.max(axis=0)  # 最大池化
            else:
                raise ValueError("pool_method 必须是 'average' 或 'max'")
            embeddings_list.append(pooled_embeddings)

            seq_index = int(os.path.basename(logits_file).split('_')[0])
            seq_indices.append(seq_index)

        except Exception as e:
            print(f"加载文件 {logits_file} 或 {embeddings_file} 时出错: {e}")
            continue

    if not logits_list or not embeddings_list:
        print("没有成功加载任何 logits 或 embeddings 文件，跳过此文件夹。\n")
        return

    # 确保所有 logits 的形状一致
    try:
        combined_logits = np.vstack(logits_list)  # 形状: [n_sequences, logits_dim]
    except ValueError as ve:
        print(f"在堆叠 logits 时出错: {ve}")
        return

    # 确保所有 embeddings 的形状一致
    try:
        combined_embeddings = np.vstack(embeddings_list)  # 形状: [n_sequences, embedding_dim]
    except ValueError as ve:
        print(f"在堆叠 embeddings 时出错: {ve}")
        return

    print(f"汇总后的 logits 形状: {combined_logits.shape}")
    print(f"汇总后的 embeddings 形状: {combined_embeddings.shape}")

    combined_logits_path = os.path.join(feature_dir, 'combined_logits.npy')
    combined_embeddings_path = os.path.join(feature_dir, 'combined_embeddings.npy')

    np.save(combined_logits_path, combined_logits)
    np.save(combined_embeddings_path, combined_embeddings)

    print(f"已保存汇总 logits 到 {combined_logits_path}")
    print(f"已保存汇总 embeddings 到 {combined_embeddings_path}")

    try:
        df_embeddings = pd.DataFrame(combined_embeddings,
                                     columns=[f'embeddings_{i}' for i in range(combined_embeddings.shape[1])])
        df_embeddings.insert(0, 'seq_index', seq_indices)
        csv_embeddings_path = os.path.join(feature_dir, 'combined_embeddings.csv')
        df_embeddings.to_csv(csv_embeddings_path, index=False, encoding='utf-8-sig')
        print(f"已保存汇总 embeddings CSV 到 {csv_embeddings_path}")
    except Exception as e:
        print(f"保存 embeddings CSV 时出错: {e}")

    try:
        if embedding_dim is not None:
            pca_components = embedding_dim
        else:
            pca_components = 50
        pca = PCA(n_components=pca_components)
        embeddings_reduced = pca.fit_transform(combined_embeddings)

        df_embeddings_reduced = pd.DataFrame(embeddings_reduced,
                                             columns=[f'embeddings_pca_{i}' for i in range(1, pca_components+1)])
        df_embeddings_reduced.insert(0, 'seq_index', seq_indices)
        csv_embeddings_reduced_path = os.path.join(feature_dir, 'embeddings_reduced.csv')
        df_embeddings_reduced.to_csv(csv_embeddings_reduced_path, index=False, encoding='utf-8-sig')
        print(f"已保存 PCA 降维后的 embeddings CSV 到 {csv_embeddings_reduced_path}\n")
    except Exception as e:
        print(f"PCA 降维或保存降维后的 embeddings 时出错: {e}\n")

# # 定义特征输出文件夹列表
# feature_dirs = [
#     '/content/drive/MyDrive/AFP_work/esmc_600_train_pos',
#     '/content/drive/MyDrive/AFP_work/esmc_600_train_neg',
#     '/content/drive/MyDrive/AFP_work/esmc_600_test_pos',  # 请确认此路径是否正确
#     '/content/drive/MyDrive/AFP_work/esmc_600_test_neg'
#     # 如果有其他文件夹，如 esmc_600_test_pos，请在此添加
# ]


# feature_dirs = [
#     'external_set_result/esmc_600_external_test/esmc_600_external_pos',
#     'external_set_result/esmc_600_external_test/esmc_600_external_neg'
# ]

# feature_dirs = [
#     'test_data_low_plddt/esmc-600/test_pos_low_plddt',
#     'test_data_low_plddt/esmc-600/test_neg_low_plddt'
# ]



# 定义特征输出文件夹列表
feature_dirs = [
    'external_set_result/esmc_600_external_test/esmc_600_external_pos',
    'external_set_result/esmc_600_external_test/esmc_600_external_neg'
]


for feature_dir in feature_dirs:
    if not os.path.isdir(feature_dir):
        print(f"文件夹 {feature_dir} 不存在，跳过。\n")
        continue
    combine_features_with_pooling(feature_dir, pool_method='average', embedding_dim=50)

print("所有文件夹的特征汇总和保存已完成。")


## 处理结构

In [ ]:
import os
import time
import json
from Bio.PDB import PDBParser
import numpy as np

# 训练集和测试集的 PDB 文件夹路径
train_pos_folder_path = 'pdb/train_pos'
train_neg_folder_path = 'pdb/train_neg'
test_pos_folder_path = 'pdb/test_pos'
test_neg_folder_path = 'pdb/test_neg'


# # low_plddt 测试集的 PDB 文件夹路径
# test_pos_folder_path = 'test_data_low_plddt/pdb/test_pos_low_plddt'
# test_neg_folder_path = 'test_data_low_plddt/pdb/test_neg_low_plddt'

# # 输出文件夹路径  11A
# output_train_pos_folder_path = 'pdb_features_11A/train_pos'
# output_train_neg_folder_path = 'pdb_features_11A/train_neg'
# output_test_pos_folder_path = 'pdb_features_11A/test_pos'
# output_test_neg_folder_path = 'pdb_features_11A/test_neg'

# # 输出文件夹路径  10A
# output_train_pos_folder_path = 'pdb_features_10A/train_pos'
# output_train_neg_folder_path = 'pdb_features_10A/train_neg'
# output_test_pos_folder_path = 'pdb_features_10A/test_pos'
# output_test_neg_folder_path = 'pdb_features_10A/test_neg'

# 输出文件夹路径   9A
output_train_pos_folder_path = 'pdb_features_9A/train_pos'
output_train_neg_folder_path = 'pdb_features_9A/train_neg'
output_test_pos_folder_path = 'pdb_features_9A/test_pos'
output_test_neg_folder_path = 'pdb_features_9A/test_neg'

# # 输出文件夹路径   8A
# output_train_pos_folder_path = 'pdb_features_8A/train_pos'
# output_train_neg_folder_path = 'pdb_features_8A/train_neg'
# output_test_pos_folder_path = 'pdb_features_8A/test_pos'
# output_test_neg_folder_path = 'pdb_features_8A/test_neg'

# # 输出文件夹路径   7A
# output_train_pos_folder_path = 'pdb_features_7A/train_pos'
# output_train_neg_folder_path = 'pdb_features_7A/train_neg'
# output_test_pos_folder_path = 'pdb_features_7A/test_pos'
# output_test_neg_folder_path = 'pdb_features_7A/test_neg'


# # 输出文件夹路径   9A  low——plddt
# output_test_pos_folder_path = 'test_data_low_plddt/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'test_data_low_plddt/pdb_features_9A/test_neg'

# # # 输出文件夹路径   9A  esmc 300m
# output_train_pos_folder_path = 'esmc_300m/pdb_features_9A/train_pos'
# output_train_neg_folder_path = 'esmc_300m/pdb_features_9A/train_neg'
# output_test_pos_folder_path = 'esmc_300m/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'esmc_300m/pdb_features_9A/test_neg'


# # 独立测试集的 PDB 文件夹路径
# test_pos_folder_path = 'external_set_result/external_pdb/external_pos_pdb'
# test_neg_folder_path = 'external_set_result/external_pdb/external_neg_pdb'

# # # # 输出文件夹路径   9A  esmc 600 external_set
# output_test_pos_folder_path = 'external_set_result/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'external_set_result/pdb_features_9A/test_neg'


# 创建输出文件夹（如果不存在）
os.makedirs(output_train_pos_folder_path, exist_ok=True)
os.makedirs(output_train_neg_folder_path, exist_ok=True)
os.makedirs(output_test_pos_folder_path, exist_ok=True)
os.makedirs(output_test_neg_folder_path, exist_ok=True)

# 提取训练集和测试集中的所有 PDB 文件
train_pos_pdb_files = [f for f in os.listdir(train_pos_folder_path) if f.endswith('.pdb') or f.endswith('.PDB')]
train_neg_pdb_files = [f for f in os.listdir(train_neg_folder_path) if f.endswith('.pdb') or f.endswith('.PDB')]
test_pos_pdb_files = [f for f in os.listdir(test_pos_folder_path) if f.endswith('.pdb') or f.endswith('.PDB')]
test_neg_pdb_files = [f for f in os.listdir(test_neg_folder_path) if f.endswith('.pdb') or f.endswith('.PDB')]

parser = PDBParser(QUIET=True)

start_time = time.time()

def process_pdb_file(pdb_path):
    try:
        structure = parser.get_structure('', pdb_path)
        residues = [residue for residue in structure.get_residues() if 'CA' in residue]
        num_residues = len(residues)

        if num_residues == 0:
            print(f" 文件 '{pdb_path}' 中没有找到 CA 原子。")
            return None, None, None, None

        # 提取位置特征、方向特征和旋转特征
        positions = np.array([residue['CA'].get_coord() for residue in residues], dtype=np.float64)
        edges = []
        directions = []
        rotations = []

        # 计算接触图和附加特征
        for i in range(num_residues):
            for j in range(i + 1, num_residues):
                distance = np.linalg.norm(positions[i] - positions[j])
                if distance < 9.0:  # 阈值为8Å来定义接触
                    edges.append([i, j])
                    direction = positions[j] - positions[i]
                    norm = np.linalg.norm(direction)
                    if norm != 0:
                        directions.append(direction / norm)
                        rotations.append(float(np.arctan2(direction[1], direction[0])))

        return positions, edges, directions, rotations
    except Exception as e:
        print(f" 处理文件 '{pdb_path}' 时出错: {e}")
        return None, None, None, None

def process_and_save(pdb_files, folder_path, output_folder_path, label):
    print(f'处理文件夹: {folder_path}')
    print(f'文件数量: {len(pdb_files)}')

    processed_count = 0
    for pdb_file in pdb_files:
        pdb_path = os.path.join(folder_path, pdb_file)
        positions, edges, directions, rotations = process_pdb_file(pdb_path)

        if positions is None:
            continue

        features = {
            "node_features": positions.tolist(),
            "edge_features": {
                "edges": edges,
                "directions": [d.tolist() for d in directions],
                "rotations": [float(rot) for rot in rotations]
            },
            "label": label
        }

        output_file_path = os.path.join(output_folder_path, f'{os.path.splitext(pdb_file)[0]}_features.json')

        try:
            with open(output_file_path, 'w') as output_file:
                json.dump(features, output_file)
            processed_count += 1
        except Exception as e:
            print(f" 保存文件 '{output_file_path}' 时出错: {e}")
            continue

        if processed_count % 100 == 0:
            print(f' 已处理 {processed_count} 个文件。')

    print(f' 完成处理 {processed_count} 个文件。')

process_and_save(train_pos_pdb_files, train_pos_folder_path, output_train_pos_folder_path, label=1)
process_and_save(train_neg_pdb_files, train_neg_folder_path, output_train_neg_folder_path, label=0)
process_and_save(test_pos_pdb_files, test_pos_folder_path, output_test_pos_folder_path, label=1)
process_and_save(test_neg_pdb_files, test_neg_folder_path, output_test_neg_folder_path, label=0)
 
end_time = time.time()
print(f"总处理时间: {end_time - start_time:.2f} 秒")
 
def count_json_files(output_folder_path):
    json_files = [f for f in os.listdir(output_folder_path) if f.endswith('.json')]
    print(f'文件夹 "{output_folder_path}" 中保存的 JSON 文件数量: {len(json_files)}')
    return json_files

print("\n保存的 JSON 文件数量:")
count_json_files(output_train_pos_folder_path)
count_json_files(output_train_neg_folder_path)
count_json_files(output_test_pos_folder_path)
count_json_files(output_test_neg_folder_path)

# 检查每个保存的 JSON 文件中的维度信息
def check_json_dimensions(output_folder_path):
    json_files = [f for f in os.listdir(output_folder_path) if f.endswith('.json')]
    for json_file in json_files[:5]:  # 仅检查前5个文件
        json_path = os.path.join(output_folder_path, json_file)
        with open(json_path, 'r') as file:
            data = json.load(file)
            node_features = data.get("node_features", [])
            edge_features = data.get("edge_features", {})
            edges = edge_features.get("edges", [])
            directions = edge_features.get("directions", [])
            rotations = edge_features.get("rotations", [])
            print(f'文件: {json_file}')
            print(f'节点特征数量: {len(node_features)}, 位置特征维度: {len(node_features[0]) if node_features else 0}')
            print(f'边特征数量: {len(edges)}, 方向特征数量: {len(directions)}, 旋转特征数量: {len(rotations)}\n')

print("\n检查部分 JSON 文件的维度信息:")
check_json_dimensions(output_train_pos_folder_path)
check_json_dimensions(output_train_neg_folder_path)
check_json_dimensions(output_test_pos_folder_path)
check_json_dimensions(output_test_neg_folder_path)


In [ ]:
import os
import glob

# # 输出文件夹路径  11A
# output_train_pos_folder_path = 'pdb_features_11A/train_pos'
# output_train_neg_folder_path = 'pdb_features_11A/train_neg'
# output_test_pos_folder_path = 'pdb_features_11A/test_pos'
# output_test_neg_folder_path = 'pdb_features_11A/test_neg'

# # 输出文件夹路径  10A
# output_train_pos_folder_path = 'pdb_features_10A/train_pos'
# output_train_neg_folder_path = 'pdb_features_10A/train_neg'
# output_test_pos_folder_path = 'pdb_features_10A/test_pos'
# output_test_neg_folder_path = 'pdb_features_10A/test_neg'

# 输出文件夹路径   9A
output_train_pos_folder_path = 'pdb_features_9A/train_pos'
output_train_neg_folder_path = 'pdb_features_9A/train_neg'
output_test_pos_folder_path = 'pdb_features_9A/test_pos'
output_test_neg_folder_path = 'pdb_features_9A/test_neg'

# # 输出文件夹路径   8A
# output_train_pos_folder_path = 'pdb_features_8A/train_pos'
# output_train_neg_folder_path = 'pdb_features_8A/train_neg'
# output_test_pos_folder_path = 'pdb_features_8A/test_pos'
# output_test_neg_folder_path = 'pdb_features_8A/test_neg'

# # 输出文件夹路径   7A
# output_train_pos_folder_path = 'pdb_features_7A/train_pos'
# output_train_neg_folder_path = 'pdb_features_7A/train_neg'
# output_test_pos_folder_path = 'pdb_features_7A/test_pos'
# output_test_neg_folder_path = 'pdb_features_7A/test_neg'

# # 输出文件夹路径  9A  low plddt
# output_test_pos_folder_path = 'test_data_low_plddt/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'test_data_low_plddt/pdb_features_9A/test_neg'

# # 输出文件夹路径   9A esmc 300m
# output_train_pos_folder_path = 'esmc_300m/pdb_features_9A/train_pos'
# output_train_neg_folder_path = 'esmc_300m/pdb_features_9A/train_neg'
# output_test_pos_folder_path = 'esmc_300m/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'esmc_300m/pdb_features_9A/test_neg'

# # 输出文件夹路径   9A esmc 300m
# output_test_pos_folder_path = 'external_set_result/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'external_set_result/pdb_features_9A/test_neg'

folders = {
    'train_pos': output_train_pos_folder_path,
    'train_neg': output_train_neg_folder_path,
    'test_pos': output_test_pos_folder_path,
    'test_neg': output_test_neg_folder_path
}

# 遍历每个文件夹并统计 JSON 文件数量
for folder_name, folder_path in folders.items():
    if os.path.isdir(folder_path):
        # 使用 glob 查找所有 .json 文件（不区分大小写）
        json_files = glob.glob(os.path.join(folder_path, '*.json')) + glob.glob(os.path.join(folder_path, '*.JSON'))
        count = len(json_files)
        print(f"文件夹 '{folder_name}' 中的 JSON 文件数量: {count}")
    else:
        print(f" 文件夹 '{folder_name}' 不存在。请检查路径是否正确。")


In [ ]:
import os
import glob
import json
import pandas as pd
from tqdm import tqdm
import concurrent.futures
import logging
import torch
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader

logging.basicConfig(filename='pdb_features_9A/processing.log',
                    filemode='a',
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    level=logging.INFO)

# # 输出文件夹路径  11A
# output_train_pos_folder_path = 'pdb_features_11A/train_pos'
# output_train_neg_folder_path = 'pdb_features_11A/train_neg'
# output_test_pos_folder_path = 'pdb_features_11A/test_pos'
# output_test_neg_folder_path = 'pdb_features_11A/test_neg'                    

# # 输出文件夹路径  10A
# output_train_pos_folder_path = 'pdb_features_10A/train_pos'
# output_train_neg_folder_path = 'pdb_features_10A/train_neg'
# output_test_pos_folder_path = 'pdb_features_10A/test_pos'
# output_test_neg_folder_path = 'pdb_features_10A/test_neg'

# 输出文件夹路径   9A
output_train_pos_folder_path = 'pdb_features_9A/train_pos'
output_train_neg_folder_path = 'pdb_features_9A/train_neg'
output_test_pos_folder_path = 'pdb_features_9A/test_pos'
output_test_neg_folder_path = 'pdb_features_9A/test_neg'

# # 输出文件夹路径   8A
# output_train_pos_folder_path = 'pdb_features_8A/train_pos'
# output_train_neg_folder_path = 'pdb_features_8A/train_neg'
# output_test_pos_folder_path = 'pdb_features_8A/test_pos'
# output_test_neg_folder_path = 'pdb_features_8A/test_neg'

# # 输出文件夹路径   7A
# output_train_pos_folder_path = 'pdb_features_7A/train_pos'
# output_train_neg_folder_path = 'pdb_features_7A/train_neg'
# output_test_pos_folder_path = 'pdb_features_7A/test_pos'
# output_test_neg_folder_path = 'pdb_features_7A/test_neg'


# # 输出文件夹路径  9A  low plddt
# output_test_pos_folder_path = 'test_data_low_plddt/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'test_data_low_plddt/pdb_features_9A/test_neg'  


# # # 输出文件夹路径  9A  esmc 300m
# output_train_pos_folder_path = 'esmc_300m/pdb_features_9A/train_pos'
# output_train_neg_folder_path = 'esmc_300m/pdb_features_9A/train_neg'
# output_test_pos_folder_path = 'esmc_300m/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'esmc_300m/pdb_features_9A/test_neg'


# # 输出文件夹路径   9A esmc 300m
# output_test_pos_folder_path = 'external_set_result/pdb_features_9A/test_pos'
# output_test_neg_folder_path = 'external_set_result/pdb_features_9A/test_neg'

output_folders = {
    'train_pos': output_train_pos_folder_path,
    'train_neg': output_train_neg_folder_path,
    'test_pos': output_test_pos_folder_path,
    'test_neg': output_test_neg_folder_path
}

for folder in output_folders.values():
    os.makedirs(folder, exist_ok=True)

def load_single_json(json_file):
    try:
        with open(json_file, 'r') as f:
            sample = json.load(f)
        logging.info(f"成功加载文件: {json_file}")
        return sample
    except Exception as e:
        logging.error(f"加载文件 '{json_file}' 时出错: {e}")
        return None

def load_json_files_parallel(folder_path, max_workers=8):
    json_files = glob.glob(os.path.join(folder_path, '*.json'))
    data = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(load_single_json, f): f for f in json_files}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc=f'Loading {os.path.basename(folder_path)}'):
            result = future.result()
            if result is not None:
                data.append(result)
    return data

train_data = []
test_data = []

train_pos_data = load_json_files_parallel(output_folders['train_pos'])
train_neg_data = load_json_files_parallel(output_folders['train_neg'])
train_data = train_pos_data + train_neg_data

test_pos_data = load_json_files_parallel(output_folders['test_pos'])
test_neg_data = load_json_files_parallel(output_folders['test_neg'])
test_data = test_pos_data + test_neg_data

print(f" 训练集总样本数: {len(train_data)}")
print(f" 测试集总样本数: {len(test_data)}")

# 定义输出汇总文件的路径
# aggregated_output_folder = 'pdb_features_11A/aggregated'
# aggregated_output_folder = 'pdb_features_10A/aggregated'
aggregated_output_folder = 'pdb_features_9A/aggregated'
# aggregated_output_folder = 'pdb_features_8A/aggregated'
# aggregated_output_folder = 'pdb_features_7A/aggregated'
# aggregated_output_folder = 'test_data_low_plddt/pdb_features_9A/aggregated'
# aggregated_output_folder = 'esmc_300m/pdb_features_9A/aggregated'
# aggregated_output_folder = 'external_set_result/pdb_features_9A/aggregated'

os.makedirs(aggregated_output_folder, exist_ok=True)

train_output_path = os.path.join(aggregated_output_folder, 'train_dataset.json')
test_output_path = os.path.join(aggregated_output_folder, 'test_dataset.json')

# 保存训练集
try:
    with open(train_output_path, 'w') as f:
        json.dump(train_data, f)
    print(f"  训练集已保存到 '{train_output_path}'。")
except Exception as e:
    print(f"  保存训练集时出错: {e}")

# 保存测试集
try:
    with open(test_output_path, 'w') as f:
        json.dump(test_data, f)
    print(f"  测试集已保存到 '{test_output_path}'。")
except Exception as e:
    print(f" 保存测试集时出错: {e}")

# 验证汇总结果
def load_aggregated_data(file_path):
    if not os.path.isfile(file_path):
        print(f"  文件 '{file_path}' 不存在。")
        return []
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        print(f"  成功加载 '{file_path}'，样本数: {len(data)}")
        return data
    except Exception as e:
        print(f"  加载文件 '{file_path}' 时出错: {e}")
        return []

# # 加载并查看训练集
train_dataset = load_aggregated_data(train_output_path)
if train_dataset:
    print(f"训练集第一个样本内容:")
    print(json.dumps(train_dataset[0], indent=2))

# 加载并查看测试集
test_dataset = load_aggregated_data(test_output_path)
if test_dataset:
    print(f"测试集第一个样本内容:")
    print(json.dumps(test_dataset[0], indent=2))

def json_to_dataframe(data):
    records = []
    for sample in data:
        record = {}
        record['label'] = sample['label']
        node_features = np.array(sample['node_features'])
        record['node_mean_x'] = node_features[:, 0].mean()
        record['node_mean_y'] = node_features[:, 1].mean()
        record['node_mean_z'] = node_features[:, 2].mean()
        record['node_std_x'] = node_features[:, 0].std()
        record['node_std_y'] = node_features[:, 1].std()
        record['node_std_z'] = node_features[:, 2].std()
        records.append(record)
    df = pd.DataFrame(records)
    return df
 
train_df = json_to_dataframe(train_dataset)
test_df = json_to_dataframe(test_dataset)

print("训练集 DataFrame 预览:")
print(train_df.head())

print("\n测试集 DataFrame 预览:")
print(test_df.head())
 
train_csv_path = os.path.join(aggregated_output_folder, 'train_dataset_dataframe.csv')
test_csv_path = os.path.join(aggregated_output_folder, 'test_dataset_dataframe.csv')
train_df.to_csv(train_csv_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_csv_path, index=False, encoding='utf-8-sig')
print(f" 训练集 DataFrame 已保存到 '{train_csv_path}'。")
print(f" 测试集 DataFrame 已保存到 '{test_csv_path}'。")

class PDBDataset(Dataset):
    def __init__(self, data_list):
        super(PDBDataset, self).__init__()
        self.data_list = data_list

    def len(self):
        return len(self.data_list)

    def get(self, idx):
        sample = self.data_list[idx]
        node_features = torch.tensor(sample['node_features'], dtype=torch.float)
        edge_index = torch.tensor(sample['edge_features']['edges'], dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(sample['edge_features']['directions'], dtype=torch.float)
        rotations = torch.tensor(sample['edge_features']['rotations'], dtype=torch.float).unsqueeze(1)
        edge_features = torch.cat([edge_attr, rotations], dim=1)  # 合并方向和旋转特征
        label = torch.tensor(sample['label'], dtype=torch.long)

        data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features, y=label)
        return data


train_pyg_dataset = PDBDataset(train_dataset)
test_pyg_dataset = PDBDataset(test_dataset)

train_loader = DataLoader(train_pyg_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_pyg_dataset, batch_size=32, shuffle=False)

print(f" PyTorch Geometric 训练集数据量: {len(train_pyg_dataset)}")
print(f" PyTorch Geometric 测试集数据量: {len(test_pyg_dataset)}")


## model

In [ ]:
import os
import numpy as np
import pandas as pd

import json
from tqdm import tqdm
import random
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader

from torch_geometric.nn import (
    GATConv, SAGEConv, GINConv, Set2Set, global_mean_pool
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, precision_recall_fscore_support,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)
import copy
import optuna
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
from sklearn.manifold import TSNE
import seaborn as sns
from torch_geometric.loader import DataLoader

# esmc_folders = {
#     'train_pos': '/content/drive/MyDrive/AFP_work/esmc_600_train_pos',
#     'train_neg': '/content/drive/MyDrive/AFP_work/esmc_600_train_neg',
#     'test_pos': '/content/drive/MyDrive/AFP_work/esmc_600_test_pos',
#     'test_neg': '/content/drive/MyDrive/AFP_work/esmc_600_test_neg'
# }

# struct_folders = {
#     'train': '/content/drive/MyDrive/AFP_work/pdb_features/aggregated/train_dataset.json',
#     'test': '/content/drive/MyDrive/AFP_work/pdb_features/aggregated/test_dataset.json'
# }

# aggregated_output_folder = '/content/drive/MyDrive/AFP_work/esmc_struct_aggregated'
# os.makedirs(aggregated_output_folder, exist_ok=True)



esmc_folders = {
    'train_pos': 'esmc_600_train_pos',
    'train_neg': 'esmc_600_train_neg',
    'test_pos': 'esmc_600_test_pos',
    'test_neg': 'esmc_600_test_neg'
}

# esmc_folders = {
#     'train_pos': 'esmc_300_train_pos',
#     'train_neg': 'esmc_300_train_neg',
#     'test_pos': 'esmc_300_test_pos',
#     'test_neg': 'esmc_300_test_neg'
# }

# struct_folders = {
#     'train': 'pdb_features_11A/aggregated/train_dataset.json',
#     'test': 'pdb_features_11A/aggregated/test_dataset.json'
# }

# struct_folders = {
#     'train': 'pdb_features_10A/aggregated/train_dataset.json',
#     'test': 'pdb_features_10A/aggregated/test_dataset.json'
# }

struct_folders = {
    'train': 'pdb_features_9A/aggregated/train_dataset.json',
    'test': 'pdb_features_9A/aggregated/test_dataset.json'
}

# struct_folders = {
#     'train': 'pdb_features_8A/aggregated/train_dataset.json',
#     'test': 'pdb_features_8A/aggregated/test_dataset.json'
# }

# struct_folders = {
#     'train': 'pdb_features_7A/aggregated/train_dataset.json',
#     'test': 'pdb_features_7A/aggregated/test_dataset.json'
# }


# struct_folders = {
#     'train': 'esmc_300m/pdb_features_9A/aggregated/train_dataset.json',
#     'test': 'esmc_300m/pdb_features_9A/aggregated/test_dataset.json'
# }


# aggregated_output_folder = 'esmc_struct_aggregated_11A_CrossAttention'
# aggregated_output_folder = 'esmc_struct_aggregated_10A_CrossAttention'
# aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention_new_heatmap'
# aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention_251121'
aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention'
# aggregated_output_folder = 'esmc_struct_aggregated_8A_CrossAttention'
# aggregated_output_folder = 'esmc_struct_aggregated_7A_CrossAttention'

# aggregated_output_folder = 'esmc_300m/esmc_struct_aggregated_9A_CrossAttention_251124'

os.makedirs(aggregated_output_folder, exist_ok=True)

#***************************1、加载 ESM-C 特征***************************
def load_esmc_features(esmc_folder):
    logits_path = os.path.join(esmc_folder, 'combined_logits.npy')
    embeddings_path = os.path.join(esmc_folder, 'combined_embeddings.npy')
    logits = np.load(logits_path, allow_pickle=True)
    embeddings = np.load(embeddings_path, allow_pickle=True)

    print(f"Logits[0] 类型: {type(logits[0])}, 值: {logits[0]}")  #  类型 <class 'numpy.ndarray'>
    print("Logits sample:", logits[0])
    print("Embeddings sample:", embeddings[0]) 
    
    logits_values = []
    for l in logits:
        forward_data = l[0] if isinstance(l, np.ndarray) else l
        sequence_tensor = forward_data.sequence
        sequence_tensor = sequence_tensor.to(device='cpu', dtype=torch.float32)
        pooled_value = sequence_tensor.mean(dim=[0, 1, 2]).item()
        logits_values.append(pooled_value)

    logits_values = np.array(logits_values, dtype=np.float32).reshape(-1, 1)
    embeddings = embeddings.astype(np.float32)
    label = 1 if 'pos' in esmc_folder else 0
    labels = np.full((logits_values.shape[0],), label)
    return logits_values, embeddings, labels
# 加载训练集和测试集的 ESM-C 特征
train_pos_logits, train_pos_embeddings, train_pos_labels = load_esmc_features(esmc_folders['train_pos'])
train_neg_logits, train_neg_embeddings, train_neg_labels = load_esmc_features(esmc_folders['train_neg'])
test_pos_logits, test_pos_embeddings, test_pos_labels = load_esmc_features(esmc_folders['test_pos'])
test_neg_logits, test_neg_embeddings, test_neg_labels = load_esmc_features(esmc_folders['test_neg'])
# 合并训练集和测试集特征
train_logits = np.vstack((train_pos_logits, train_neg_logits))
train_embeddings = np.vstack((train_pos_embeddings, train_neg_embeddings))
train_labels = np.hstack((train_pos_labels, train_neg_labels))

test_logits = np.vstack((test_pos_logits, test_neg_logits))
test_embeddings = np.vstack((test_pos_embeddings, test_neg_embeddings))
test_labels = np.hstack((test_pos_labels, test_neg_labels))

print(f"训练集 logits 形状: {train_logits.shape}")  # （2400,1）
print(f"训练集 embeddings 形状: {train_embeddings.shape}") # （2400,1152）
print(f"训练集 labels 形状: {train_labels.shape}") # （2400,）

print(f"测试集 logits 形状: {test_logits.shape}") # （616,1）
print(f"测试集 embeddings 形状: {test_embeddings.shape}")  # （616,1152）
print(f"测试集 labels 形状: {test_labels.shape}") # （616,）


#***************************2、加载结构特征***************************
def load_struct_features(json_path, sample_limit=5):
    with open(json_path, 'r') as f:
        json_data = json.load(f)
    data_list = []
    for idx, sample in enumerate(tqdm(json_data, desc=f'加载结构特征 from {json_path}')):
        required_keys = ['node_features', 'edge_features', 'label']
        if not all(key in sample for key in required_keys):
            print(f"[ERROR] 样本缺少必要的键: {sample}")
            continue
        node_features = sample['node_features']
        edge_features = sample['edge_features']
        label = sample['label']
        edges = edge_features.get('edges', [])
        directions = edge_features.get('directions', [])
        rotations = edge_features.get('rotations', [])
        num_edges = len(edges)
        if not (len(directions) == num_edges and len(rotations) == num_edges):
            print(f"[ERROR] 边的数量与方向或旋转数量不匹配: {sample}")
            continue
        if num_edges > 0:
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            directions = torch.tensor(directions, dtype=torch.float)
            rotations = torch.tensor(rotations, dtype=torch.float).unsqueeze(1)
            edge_attr = torch.cat([directions, rotations], dim=1)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, 4), dtype=torch.float)
        node_features = torch.tensor(node_features, dtype=torch.float)
        label = torch.tensor(label, dtype=torch.long)
        data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=label)
        data_list.append(data)
        if idx < sample_limit:
            num_nodes = node_features.shape[0]
            node_feature_dim = node_features.shape[1]
            print(f"样本 {idx+1}: 节点数量: {num_nodes}, 节点特征维度: {node_feature_dim}, 边数量: {num_edges}")
            if num_edges > 0:
                print(f"  边特征维度: {edge_attr.shape[1]}")
            print("-" * 50)
    unique_node_feature_dims = set([data.x.shape[1] for data in data_list])
    unique_edge_feature_dims = set([data.edge_attr.shape[1] for data in data_list if data.edge_attr.shape[0] > 0])
    print(f"\n所有样本中唯一的节点特征维度: {unique_node_feature_dims}")  # 3
    print(f"所有样本中唯一的边特征维度: {unique_edge_feature_dims}")  # 4
    return data_list

train_struct_data = load_struct_features(struct_folders['train'])
test_struct_data = load_struct_features(struct_folders['test'])
# 打印一些节点特征和边特征
for i in range(min(3, len(train_struct_data))):
    data = train_struct_data[i]
    print(f"样本 {i+1} - 节点特征: {data.x.shape}, 边特征: {data.edge_attr.shape}")

##*************************** 特征标准化 ***************************
def normalize_features(train_data_list, test_data_list=None):
    node_scaler = StandardScaler()
    edge_scaler = StandardScaler()
    all_node_features = np.concatenate([data.x.numpy() for data in train_data_list], axis=0)
    all_edge_features = np.concatenate([data.edge_attr.numpy() for data in train_data_list if data.edge_attr.shape[0] > 0], axis=0)
    node_scaler.fit(all_node_features)
    if all_edge_features.size > 0:
        edge_scaler.fit(all_edge_features)
    for data in train_data_list:
        data.x = torch.tensor(node_scaler.transform(data.x.numpy()), dtype=torch.float)
        if data.edge_attr.shape[0] > 0:
            data.edge_attr = torch.tensor(edge_scaler.transform(data.edge_attr.numpy()), dtype=torch.float)
    if test_data_list:
        for data in test_data_list:
            data.x = torch.tensor(node_scaler.transform(data.x.numpy()), dtype=torch.float)
            if data.edge_attr.shape[0] > 0:
                data.edge_attr = torch.tensor(edge_scaler.transform(data.edge_attr.numpy()), dtype=torch.float)
    return train_data_list, test_data_list

train_struct_data, test_struct_data = normalize_features(train_struct_data, test_struct_data)

# #*************************** 整合 ESM-C 特征 ***************************
def integrate_features(data_list, embeddings, logits):
    if len(data_list) != len(embeddings) or len(data_list) != len(logits):
        raise ValueError(f"data_list, embeddings 和 logits 长度不匹配: {len(data_list)} vs {len(embeddings)} vs {len(logits)}")
    for i, data in enumerate(tqdm(data_list, desc='整合 ESM-C embeddings 和 logits')):
        embedding = torch.tensor(embeddings[i], dtype=torch.float)  # [1152]
        logit = torch.tensor(logits[i], dtype=torch.float).squeeze()  # [1] -> 标量
        combined_feature = torch.cat([embedding, logit.unsqueeze(0)], dim=0)  # [1152 + 1 = 1153]
        num_nodes = data.x.shape[0]
        combined_expanded = combined_feature.unsqueeze(0).repeat(num_nodes, 1)  # [num_nodes, 1153]
        data.x = torch.cat([data.x, combined_expanded], dim=1)  # [num_nodes, 3 + 1153 = 1156]
    return data_list

train_struct_data = integrate_features(train_struct_data, train_embeddings, train_logits)
test_struct_data = integrate_features(test_struct_data, test_embeddings, test_logits)

print(f"训练集第一个样本的节点特征维度（整合后）: {train_struct_data[0].x.shape[1]}")  # 1156
print(f"测试集第一个样本的节点特征维度（整合后）: {test_struct_data[0].x.shape[1]}") # 1156



# #***************************创建数据集和数据加载器**#***************************
class ProteinDataset(Dataset):
    def __init__(self, data_list):
        super(ProteinDataset, self).__init__()
        self.data_list = data_list
    def len(self):
        return len(self.data_list)
    def get(self, idx):
        return self.data_list[idx]

train_dataset = ProteinDataset(train_struct_data)
test_dataset = ProteinDataset(test_struct_data)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

class DeepGATModel(nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim, hidden_dim, out_dim, num_heads=4, dropout=0.3, num_layers=3):
        super(DeepGATModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        self.edge_preprocess = nn.Sequential(
            nn.Linear(edge_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        for layer in range(num_layers):
            in_dim = node_feature_dim if layer == 0 else hidden_dim * num_heads
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=hidden_dim,
                heads=num_heads,
                dropout=dropout,
                edge_dim=hidden_dim,  # 调整为预处理后的边特征维度
                add_self_loops=True  # 添加自环，增强稳定性。能提升稳定性（每个节点至少保留自身信息）
            ))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim * num_heads))

        # 替换 Set2Set 为更简单的池化
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim * num_heads, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        if edge_attr is not None:
            edge_attr = self.edge_preprocess(edge_attr)

        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)

        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        if edge_attr is not None:
            edge_attr = self.edge_preprocess(edge_attr)

        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)

        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GraphSAGEModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GraphSAGEModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        self.convs.append(SAGEConv(node_feature_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GINModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GINModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        for layer in range(num_layers):
            if layer == 0:
                nn_lin = nn.Sequential(nn.Linear(node_feature_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            else:
                nn_lin = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.convs.append(GINConv(nn_lin))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

# 三向交叉注意力 A学习BC
class CrossAttentionFusion(nn.Module):
    def __init__(self, feature_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.attn = nn.MultiheadAttention(
            embed_dim=feature_dim, 
            num_heads=num_heads, 
            dropout=dropout,
            # batch_first=False  # [L,B,E]
            batch_first=True  # [B,L,E]
        )
        self.norm1 = nn.LayerNorm(feature_dim)
        self.norm2 = nn.LayerNorm(feature_dim)
        self.norm3 = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)

        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)  # 可学习权重
        self.fc = nn.Linear(feature_dim, 2)

    def forward(self, features_list):
        feat_gat, feat_sage, feat_gin = features_list
        self.attn_maps = {}

        def cross(q_src, kv_srcs, norm_layer, name):
            # 构造 QKV
            q = q_src.unsqueeze(1)              # [1, B, d]  改成了  [B,1,d]
            kv = torch.stack(kv_srcs, dim=1)    # [2, B, d] 改成了  [B,2,d]

            # 计算注意力
            attn_out, attn_weights = self.attn(q, kv, kv)  # attn_out: [B, 1, d], attn_weights: [B, 1, 2]
            attn_out = attn_out.squeeze(1)                 # [B, d]
            
            # 残差连接 + LayerNorm
            out = q_src + self.dropout(attn_out)
            out = norm_layer(out)

            # 保存注意力权重（平均所有样本）
            avg_attn = attn_weights.mean(dim=0).squeeze(0).detach().cpu().numpy()  # attn_weights 的形状是 [B, 1, 2]    mean(dim=0) 得到 [1, 2]，再 squeeze(0) 得 [2]
            self.attn_maps[name] = avg_attn  # 保存到模块属性

            return out

        # 三向交叉注意力
        out_gat  = cross(feat_gat,  [feat_sage, feat_gin], self.norm1, "GAT")
        out_sage = cross(feat_sage, [feat_gat, feat_gin], self.norm2, "SAGE")
        out_gin  = cross(feat_gin,  [feat_gat, feat_sage], self.norm3, "GIN")

        # 可学习权重融合（替代简单的mean）
        weights = torch.softmax(self.fusion_weights, dim=0)
        fused = (out_gat * weights[0] + 
                out_sage * weights[1] + 
                out_gin * weights[2])

        return self.fc(fused)

# 3x3
def plot_attention_matrix(fusion_module, output_folder, epoch, mode="cross"):
    os.makedirs(output_folder, exist_ok=True)
    weights = np.zeros((3, 3))
    gat = fusion_module.attn_maps.get("GAT", [0, 0])
    sage = fusion_module.attn_maps.get("SAGE", [0, 0])
    gin = fusion_module.attn_maps.get("GIN", [0, 0])
    weights[0, 1:] = gat
    weights[1, [0, 2]] = sage
    weights[2, :2] = gin

    # # 可选：是否保留自注意力
    # if mode == "self":
    #     diag_vals = torch.softmax(fusion_module.fusion_weights, dim=0).detach().cpu().numpy()
    #     np.fill_diagonal(weights, diag_vals)

    # 设置对角线为空白   下面sns.heatmap 更改为annot=annot
    annot = weights.copy().astype(object)

    for i in range(3):
        for j in range(3):
            if i == j:
                annot[i][j] = ""        # 对角线留空
            else:
                annot[i][j] = f"{weights[i][j]:.2f}"  # 数字格式化为两位小数

    if mode == "self":
        diag_vals = torch.softmax(fusion_module.fusion_weights, dim=0).detach().cpu().numpy()
        for i in range(3):
            annot[i][i] = f"{diag_vals[i]:.2f}"

    sns.heatmap(weights, annot=annot, fmt="", cmap="YlGnBu",
                xticklabels=["GAT", "SAGE", "GIN"],
                yticklabels=["GAT", "SAGE", "GIN"],
                vmin=0, vmax=1)
    plt.title(f"Attention Map (Epoch {epoch}, mode={mode})")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f"attn_epoch_{epoch}_{mode}.png"), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# CKA分析
def linear_cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)
    Gx = X.T @ X
    Gy = Y.T @ Y
    numerator = torch.norm(X.T @ Y, p='fro') ** 2
    denom = torch.norm(Gx, p='fro') * torch.norm(Gy, p='fro')
    return (numerator / denom).item()

def plot_cka_heatmap(cka_matrix, output_folder):
    labels = ["GAT", "SAGE", "GIN"]

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cka_matrix,
        annot=True,
        fmt=".3f",
        cmap="YlGnBu",
        xticklabels=labels,
        yticklabels=labels,
        vmin=0,
        vmax=1
    )
    plt.title("CKA Similarity Matrix of GNN Branches", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "cka_heatmap.png"), dpi=600)
    plt.close()


def collect_embeddings(model, loader, device):
    model.eval()
    all_features = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            feat = model.get_last_layer_features(data)   # shape [batch, 256]
            all_features.append(feat.cpu())
    return torch.cat(all_features, dim=0)   # [N, 256]


def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path='best_model.pth'):
    best_test_acc = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler.step()
        train_acc, _, _ = test(model, train_loader, device)
        test_acc, test_trues, test_preds = test(model, test_loader, device)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break
    model.load_state_dict(best_model_wts)
    return best_test_acc, best_model_wts

def test(model, loader, device):
    model.eval()
    correct = 0
    preds, trues = [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='评估'):
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(data.y.cpu().numpy())
            correct += (pred == data.y).sum().item()
    accuracy = correct / len(loader.dataset)
    return accuracy, trues, preds

from sklearn.metrics import matthews_corrcoef, roc_auc_score, confusion_matrix

def detailed_test(model, loader, device, models=None):
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='详细评估'):
            data = data.to(device)
            if isinstance(model, CrossAttentionFusion):
                if models is None:
                    raise ValueError("models dictionary required for CrossAttentionFusion evaluation")
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
                out = model(features_list)
            else:
                out = model(data)
            prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            pred = out.argmax(dim=1).cpu().numpy()
            true = data.y.cpu().numpy()
            preds.extend(pred)
            trues.extend(true)
            probs.extend(prob)
    acc = accuracy_score(trues, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(trues, preds, average='binary')
    mcc = matthews_corrcoef(trues, preds)
    auc = roc_auc_score(trues, probs)
    tn, fp, fn, tp = confusion_matrix(trues, preds).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics = {'acc': acc, 'mcc': mcc, 'auc': auc, 'sn': sn, 'sp': sp, 'precision': precision, 'recall': recall, 'f1': f1}
    return metrics

def optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=5):
    def objective(trial):
        hidden_dim = trial.suggest_int('hidden_dim', 64, 512)
        num_layers = trial.suggest_int('num_layers', 2, 6)
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)

        if model_class == DeepGATModel:
            num_heads = trial.suggest_int('num_heads', 2, 16)
            model = DeepGATModel(
                node_feature_dim=model_params['node_feature_dim'],
                edge_feature_dim=model_params['edge_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_heads=num_heads,
                dropout=dropout,
                num_layers=num_layers
            ).to(device)
        elif model_class == GraphSAGEModel:
            model = GraphSAGEModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)
        elif model_class == GINModel:
            model = GINModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

        best_acc, _ = train_model(
            model, train_loader, test_loader, criterion, optimizer, scheduler,
            device, num_epochs=50, patience=10, model_save_path=f"best_{model_class.__name__}.pth"
        )
        return best_acc

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=5)
    print(f"总试验次数: {len(study.trials)}")
    print(f"{model_class.__name__} 最佳超参数: {study.best_params}")
    for trial in study.trials:
      print(f"Trial {trial.number}: State={trial.state}, Value={trial.value}")
    return study.best_params

def explain_features(model, test_loader, device, output_folder):
    print("正在进行 ESM-C 特征的 SHAP 分析...")
    esmc_features = np.hstack([train_embeddings, train_logits])
    labels = train_labels
    proxy_model = XGBClassifier()
    proxy_model.fit(esmc_features, labels)
    explainer = shap.Explainer(proxy_model)
    shap_values = explainer(esmc_features)
    shap.summary_plot(shap_values, esmc_features, plot_type="bar", show=False)
    plt.title("ESM-C 特征重要性 (SHAP)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "shap_esmc_features.png"), dpi=600)
    plt.close()
    print("SHAP 分析完成，结果已保存至 shap_esmc_features.png")

    print("正在进行 GNNExplainer 分析...")
    trained_model = models['DeepGATModel']
    explainer = GNNExplainer(trained_model, epochs=200, lr=0.01)
    for sample_idx in range(min(5, len(test_struct_data))):
        data = test_struct_data[sample_idx].to(device)
        node_idx = 0
        node_feat_mask, edge_mask = explainer.explain_node(node_idx, data.x, data.edge_index, data.edge_attr)
        print(f"样本 {sample_idx+1} | 节点 0 特征重要性（前5个）: {node_feat_mask[:5]} | 边重要性（前5个）: {edge_mask[:5]}")
    print("GNNExplainer 分析完成")

    print("正在进行 t-SNE 可视化...")
    def get_last_layer_features(model, loader, device):
        model.eval()
        features = []
        labels = []
        with torch.no_grad():
            for data in loader:
                data = data.to(device)
                feat = model.get_last_layer_features(data)
                features.append(feat.cpu().numpy())
                labels.append(data.y.cpu().numpy())
        return np.vstack(features), np.hstack(labels)

    features, labels = get_last_layer_features(trained_model, test_loader, device)
    tsne = TSNE(n_components=2, random_state=42)
    features_2d = tsne.fit_transform(features)
    plt.figure(figsize=(8, 6))
    plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels, cmap='coolwarm', alpha=0.6)
    plt.title("t-SNE of Last Layer Features (DeepGATModel)")
    plt.colorbar(label='Class')
    plt.savefig(os.path.join(output_folder, "tsne_last_layer.png"), dpi=600)
    plt.close()
    print("t-SNE 可视化完成，结果已保存至 tsne_last_layer.png")

    return results, models

def train_and_evaluate_models(train_loader, test_loader, device, output_folder):
    model_params = {"node_feature_dim": 1156, "edge_feature_dim": 4, "out_dim": 2}
    # model_params = {"node_feature_dim": 964, "edge_feature_dim": 4, "out_dim": 2}  #  esmc 300 embedding=960
    best_params = {}

    for model_class in [DeepGATModel, GraphSAGEModel, GINModel]:
        print(f"优化 {model_class.__name__}...")
        best_params[model_class.__name__] = optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=10)

    # 初始化模型
    models = {
        "DeepGATModel": DeepGATModel(
            node_feature_dim=1156, edge_feature_dim=4, out_dim=2,
            hidden_dim=best_params["DeepGATModel"]["hidden_dim"],
            num_layers=best_params["DeepGATModel"]["num_layers"],
            dropout=best_params["DeepGATModel"]["dropout"],
            num_heads=best_params["DeepGATModel"]["num_heads"]
        ).to(device),
        "GraphSAGEModel": GraphSAGEModel(
            node_feature_dim=1156, out_dim=2,
            hidden_dim=best_params["GraphSAGEModel"]["hidden_dim"],
            num_layers=best_params["GraphSAGEModel"]["num_layers"],
            dropout=best_params["GraphSAGEModel"]["dropout"]
        ).to(device),
        "GINModel": GINModel(
            node_feature_dim=1156, out_dim=2,
            hidden_dim=best_params["GINModel"]["hidden_dim"],
            num_layers=best_params["GINModel"]["num_layers"],
            dropout=best_params["GINModel"]["dropout"]
        ).to(device)
    }

    results = {}
    for name, model in models.items():
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=best_params[name]["lr"], weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        save_path = os.path.join(output_folder, f"best_{name}.pth")
        best_acc, _ = train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path=save_path)
        metrics = detailed_test(model, test_loader, device)
        results[name] = metrics
        print(f"{name} - Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}")

    print("\n### 三个模型性能对比 ###")
    for name, metrics in results.items():
        print(f"{name}: Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}")

    for name, model in models.items():
        model.load_state_dict(torch.load(os.path.join(output_folder, f"best_{name}.pth")))
        model.eval()

    print("\n===== Computing CKA Similarity Between GNN Models =====")

    gin_feats  = collect_embeddings(models["GINModel"],  test_loader, device)
    gat_feats  = collect_embeddings(models["DeepGATModel"], test_loader, device)
    sage_feats = collect_embeddings(models["GraphSAGEModel"], test_loader, device)

    cka_gin_gat   = linear_cka(gin_feats, gat_feats)
    cka_gin_sage  = linear_cka(gin_feats, sage_feats)
    cka_gat_sage  = linear_cka(gat_feats, sage_feats)

    print(f"CKA(GIN, GAT)  = {cka_gin_gat:.4f}")
    print(f"CKA(GIN, SAGE) = {cka_gin_sage:.4f}")
    print(f"CKA(GAT, SAGE) = {cka_gat_sage:.4f}")

    cka_matrix = np.array([
        [1.0,          cka_gat_sage, cka_gin_gat],
        [cka_gat_sage, 1.0,          cka_gin_sage],
        [cka_gin_gat,  cka_gin_sage, 1.0]
    ])

    np.savetxt(os.path.join(output_folder, "cka_matrix.csv"), cka_matrix, delimiter=",")
    plot_cka_heatmap(cka_matrix, output_folder)

    return results, models

# ### 交叉注意力融合训练
# 在交叉注意力融合训练函数中修改
def train_cross_attention_fusion(models, train_loader, test_loader, device, output_folder, num_epochs=50, patience=10):
    fusion_module = CrossAttentionFusion(feature_dim=256, num_heads=4, dropout=0.1).to(device)
    optimizer_fusion = torch.optim.Adam(fusion_module.parameters(), lr=0.001, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler_fusion = torch.optim.lr_scheduler.StepLR(optimizer_fusion, step_size=10, gamma=0.1)

    print("\n### 训练交叉注意力融合模块 ###")
    best_fusion_acc = 0
    best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        fusion_module.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'融合训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            with torch.no_grad():
                feat_gat = models['DeepGATModel'].get_last_layer_features(data).detach()
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data).detach()
                feat_gin = models['GINModel'].get_last_layer_features(data).detach()

                print(f"feat_gat shape: {feat_gat.shape}, type: {type(feat_gat)}")
                print(f"feat_sage shape: {feat_sage.shape}, type: {type(feat_sage)}")
                print(f"feat_gin shape: {feat_gin.shape}, type: {type(feat_gin)}")


                if isinstance(feat_gat, torch.Tensor) and feat_gat.dim() == 2:  # 确保是 [batch_size, feature_dim]
                    features_list = [feat_gat, feat_sage, feat_gin]
                else:
                    raise ValueError("Expected pure tensors from get_last_layer_features, got unexpected type or shape")
            out = fusion_module(features_list)
            loss = criterion(out, data.y)
            optimizer_fusion.zero_grad()
            loss.backward()
            optimizer_fusion.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler_fusion.step()

        fusion_module.eval()
        preds, trues = [], []
        with torch.no_grad():
            for data in test_loader:
                data = data.to(device)
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                # 同样的检查和处理
                if isinstance(feat_gat, torch.Tensor) and feat_gat.dim() == 2:
                    features_list = [feat_gat, feat_sage, feat_gin]
                else:
                    raise ValueError("Expected pure tensors from get_last_layer_features in test loop")
                out = fusion_module(features_list)
                pred = out.argmax(dim=1)
                preds.extend(pred.cpu().numpy())
                trues.extend(data.y.cpu().numpy())
        test_acc = accuracy_score(trues, preds)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Fusion Test Acc: {test_acc:.4f}")

        # 每5轮或最后一轮保存注意力热力图
        if (epoch % 5 == 0) or (epoch == num_epochs):
            plot_attention_matrix(fusion_module, output_folder, epoch, mode="cross")
            plot_attention_matrix(fusion_module, output_folder, epoch, mode="self")

        if test_acc > best_fusion_acc:
            best_fusion_acc = test_acc
            best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
            epochs_no_improve = 0
            torch.save(fusion_module.state_dict(), os.path.join(output_folder, "best_cross_attention_fusion.pth"))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break

    fusion_module.load_state_dict(best_fusion_wts)
    fusion_metrics = detailed_test(fusion_module, test_loader, device, models=models)  # 传递 models 参数
    return fusion_module, fusion_metrics

In [ ]:
def analyze_feature_correlations(train_struct_data, train_labels, output_folder):

    all_node_features = np.vstack([data.x.numpy() for data in train_struct_data])
    all_edge_features = np.vstack([data.edge_attr.numpy() for data in train_struct_data if data.edge_attr.size(0) > 0])
    labels = np.hstack([data.y.numpy() for data in train_struct_data])
    node_labels = np.repeat(labels, [data.x.shape[0] for data in train_struct_data])

    node_df = pd.DataFrame(all_node_features[:, :10], columns=[f"node_feat_{i}" for i in range(10)])  # 取前10个特征示例
    node_df['label'] = node_labels
    corr_matrix = node_df.corr()

    plt.figure(figsize=(15, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title("Node Feature Correlation Heatmap")
    plt.savefig(os.path.join(output_folder, "node_feature_correlation.png"), dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()

    if all_edge_features.size > 0:
        edge_df = pd.DataFrame(all_edge_features, columns=['dir_x', 'dir_y', 'dir_z', 'rotation'])
        corr_matrix = edge_df.corr()

        plt.figure(figsize=(8, 6))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
        plt.title("Edge Feature Correlation Heatmap")
        plt.savefig(os.path.join(output_folder, "edge_feature_correlation.png"), dpi=600, bbox_inches='tight', facecolor='white')
        plt.close()

    print("特征相关性分析完成，结果已保存至输出文件夹")

In [ ]:
import time, matplotlib.pyplot as plt

def analyze_long_chain_performance(models, fusion_module, test_struct_data, device, output_folder, length_threshold=50):
    
    print("\n===== 长序列样本性能分析（Long-chain peptide performance） =====")

    long_samples = [d for d in test_struct_data if d.x.shape[0] > length_threshold]
    if not long_samples:
        print(f" 未找到节点数 > {length_threshold} 的样本。")
        return

    long_loader = DataLoader(ProteinDataset(long_samples), batch_size=1, shuffle=False)
    lengths, times = [], []

    fusion_module.eval()
    torch.cuda.empty_cache()
    with torch.no_grad():
        for data in long_loader:
            data = data.to(device)
            start = time.time()
            feats = [m.get_last_layer_features(data) for m in models.values()]
            _ = fusion_module(feats)
            torch.cuda.synchronize()
            end = time.time()
            lengths.append(data.x.shape[0])
            times.append(end - start)

    df = pd.DataFrame({"length": lengths, "inference_time (s)": times})
    df.to_csv(os.path.join(output_folder, "test_long_chain_performance_metrics.csv"), index=False)
    plt.scatter(lengths, times, c="steelblue")
    plt.xlabel("Sequence Length (Residues)")
    plt.ylabel("Inference Time (s)")
    plt.title("Long-chain Peptide Inference Time Scaling")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "test_long_chain_scaling.png"), dpi=600)
    plt.close()
    print(" 长序列性能分析完成。")


In [ ]:
def analyze_test_performance(models, fusion_module, test_struct_data, test_loader, device, output_folder):
    import time, psutil, matplotlib.pyplot as plt, seaborn as sns
    from sklearn.metrics import confusion_matrix

    print("\n===== 测试集性能分析（Inference Time, Memory, GPU） =====")
    torch.cuda.empty_cache()

    fusion_module.eval()
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            feats = [m.get_last_layer_features(data) for m in models.values()]
            _ = fusion_module(feats)
    torch.cuda.synchronize()
    total_time = time.time() - start

    avg_time_per_batch = total_time / len(test_loader)
    avg_time_per_sample = total_time / len(test_struct_data)

    gpu_alloc = torch.cuda.memory_allocated(device) / 1024**2
    gpu_reserved = torch.cuda.memory_reserved(device) / 1024**2
    cpu_mem = psutil.Process().memory_info().rss / 1024**2

    perf = {
        "total_inference_time (s)": total_time,
        "avg_inference_time_per_batch (s)": avg_time_per_batch,
        "avg_inference_time_per_sample (s)": avg_time_per_sample,
        "gpu_memory_allocated (MB)": gpu_alloc,
        "gpu_memory_reserved (MB)": gpu_reserved,
        "cpu_memory (MB)": cpu_mem
    }
    pd.DataFrame([perf]).to_csv(os.path.join(output_folder, "final_test_performance_metrics.csv"), index=False)
    print(" 性能指标已保存。")

    analyze_long_chain_performance(models, fusion_module, test_struct_data, device, output_folder)

    print("\n===== 混淆矩阵分析 =====")
    trues, preds = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            feats = [m.get_last_layer_features(data) for m in models.values()]
            out = fusion_module(feats)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            trues.extend(data.y.cpu().numpy())
    cm = confusion_matrix(trues, preds)
    cm_df = pd.DataFrame(cm, index=["True_0", "True_1"], columns=["Pred_0", "Pred_1"])
    cm_df.to_csv(os.path.join(output_folder, "final_test_confusion_matrix.csv"), index=False)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix (Test Set)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "final_test_confusion_matrix.png"), dpi=600)
    plt.close()
    print(" 混淆矩阵图已保存。")


In [ ]:
def evaluate_on_testset(train_struct_data, test_struct_data, device, output_folder):
    print("\n===== 使用完整训练集重新训练，并在测试集上评估 =====")

    train_loader = DataLoader(ProteinDataset(train_struct_data), batch_size=32, shuffle=True)
    test_loader = DataLoader(ProteinDataset(test_struct_data), batch_size=32, shuffle=False)

    results, models = train_and_evaluate_models(train_loader, test_loader, device, output_folder)
    fusion_module, fusion_metrics = train_cross_attention_fusion(models, train_loader, test_loader, device, output_folder)
    results["CrossAttentionFusion"] = fusion_metrics

    print(f"\n Fusion Model 测试集表现: "
          f"Acc={fusion_metrics['acc']:.4f}, MCC={fusion_metrics['mcc']:.4f}, "
          f"AUC={fusion_metrics['auc']:.4f}, F1={fusion_metrics['f1']:.4f}")

    analyze_test_performance(models, fusion_module, test_struct_data, test_loader, device, output_folder)

    metrics_list = []
    for model_name, metrics in results.items():
        metrics_entry = {"model": model_name}
        metrics_entry.update(metrics)
        metrics_list.append(metrics_entry)

    df = pd.DataFrame(metrics_list)
    csv_path = os.path.join(output_folder, "testset_evaluation_results.csv")
    df.to_csv(csv_path, index=False)
    print(f"\n 所有模型的测试集指标结果已保存至: {csv_path}")

    return results, fusion_metrics

In [ ]:
from sklearn.model_selection import StratifiedKFold
def cross_validation_on_trainset(train_struct_data, train_labels, device, output_folder, k_folds=5):

    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    all_metrics = []

    total_samples = len(train_labels)
    print(f"\n===== 开始 {k_folds}-折交叉验证 =====")
    print(f" 总样本数: {total_samples}")
    print(f" 每折平均样本数: {total_samples // k_folds}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(train_labels)), train_labels)):
        print(f"\n===== Fold {fold+1}/{k_folds} =====")

        train_data = [train_struct_data[i] for i in train_idx]
        val_data = [train_struct_data[i] for i in val_idx]
        train_loader = DataLoader(ProteinDataset(train_data), batch_size=32, shuffle=True)
        val_loader = DataLoader(ProteinDataset(val_data), batch_size=32, shuffle=False)

        y_train, y_val = train_labels[train_idx], train_labels[val_idx]
        num_train, num_val = len(train_idx), len(val_idx)
        num_pos_train = np.sum(y_train)
        num_neg_train = num_train - num_pos_train
        num_pos_val = np.sum(y_val)
        num_neg_val = num_val - num_pos_val

        print(f" 当前折样本划分：")
        print(f"  - 训练集样本数: {num_train}  (正样本 {num_pos_train}, 负样本 {num_neg_train})")
        print(f"  - 验证集样本数: {num_val}  (正样本 {num_pos_val}, 负样本 {num_neg_val})")

        results, models = train_and_evaluate_models(train_loader, val_loader, device, output_folder)
        fusion_module, fusion_metrics = train_cross_attention_fusion(models, train_loader, val_loader, device, output_folder)
        results["CrossAttentionFusion"] = fusion_metrics

        for name, metrics in results.items():
            all_metrics.append({
                "fold": fold+1, "model": name,
                **metrics
            })

    df = pd.DataFrame(all_metrics)
    df_mean = df.groupby("model").mean(numeric_only=True).reset_index()
    df_mean["fold"] = "mean"
    df = pd.concat([df, df_mean], ignore_index=True)
    csv_path = os.path.join(output_folder, "five_fold_cross_validation_results.csv")
    df.to_csv(csv_path, index=False)
    print(f" 五折交叉验证完成，结果保存至 {csv_path}")
    return df


In [19]:
import sys, os, datetime

def setup_jupyter_logger(output_folder, log_prefix="train"):
    """
    在 Jupyter Notebook 中保存控制台输出日志。
    所有 print() 内容同时写入文件 + 正常显示在 notebook。
    """
    os.makedirs(output_folder, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = os.path.join(output_folder, f"{log_prefix}_log_{timestamp}.log")

    class JupyterLogger(object):
        def __init__(self, filename):
            self.terminal = sys.__stdout__    # 保留 Jupyter 输出
            self.log = open(filename, "a", encoding="utf-8")
        def write(self, message):
            self.terminal.write(message)
            self.log.write(message)
            self.log.flush()
        def flush(self):
            self.terminal.flush()
            self.log.flush()

    sys.stdout = JupyterLogger(log_path)
    sys.stderr = sys.stdout

    print("=" * 80)
    print(f" 实验启动时间: {timestamp}")
    print(f" 输出目录: {output_folder}")
    print("=" * 80)

    return log_path


In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    log_path = setup_jupyter_logger(aggregated_output_folder, log_prefix="AFP_train")
    print(f" 日志记录开始，输出文件：{log_path}")

    # 执行五折交叉验证（仅训练集）
    df_cv = cross_validation_on_trainset(train_struct_data, train_labels, device, aggregated_output_folder, k_folds=5)

#    # 在完整训练集上评估测试集表现
#     test_results, fusion_metrics = evaluate_on_testset(train_struct_data, test_struct_data, device, aggregated_output_folder)

    print("=" * 80)
    print(f" 实验全部完成！日志文件保存路径: {log_path}")
    print("=" * 80)


## 消融实验
1、仅使用ESMC特征；
2、仅使用结构特征

### 仅使用ESMC特征

In [ ]:
import os
import numpy as np
import json
from tqdm import tqdm
import random
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import (
    GATConv, SAGEConv, GINConv, global_mean_pool
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, matthews_corrcoef,
    roc_auc_score, confusion_matrix
)
import copy
import optuna

# 文件路径
# esmc_folders = {
#     'train_pos': '/content/drive/MyDrive/AFP_work/esmc_600_train_pos',
#     'train_neg': '/content/drive/MyDrive/AFP_work/esmc_600_train_neg',
#     'test_pos': '/content/drive/MyDrive/AFP_work/esmc_600_test_pos',
#     'test_neg': '/content/drive/MyDrive/AFP_work/esmc_600_test_neg'
# }

# struct_folders = {
#     'train': '/content/drive/MyDrive/AFP_work/pdb_features/aggregated/train_dataset.json',
#     'test': '/content/drive/MyDrive/AFP_work/pdb_features/aggregated/test_dataset.json'
# }


esmc_folders = {
    'train_pos': 'esmc_600_train_pos',
    'train_neg': 'esmc_600_train_neg',
    'test_pos': 'esmc_600_test_pos',
    'test_neg': 'esmc_600_test_neg'
}

struct_folders = {
    'train': 'pdb_features_9A/aggregated/train_dataset.json',
    'test': 'pdb_features_9A/aggregated/test_dataset.json'
}

aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention/Ablation experiments /only_esmc'
os.makedirs(aggregated_output_folder, exist_ok=True)


def load_esmc_features(esmc_folder):
    logits_path = os.path.join(esmc_folder, 'combined_logits.npy')
    embeddings_path = os.path.join(esmc_folder, 'combined_embeddings.npy')
    logits = np.load(logits_path, allow_pickle=True)
    embeddings = np.load(embeddings_path, allow_pickle=True)
    logits_values = []
    for l in logits:
        forward_data = l[0] if isinstance(l, np.ndarray) else l
        sequence_tensor = forward_data.sequence.to(device='cpu', dtype=torch.float32)
        pooled_value = sequence_tensor.mean(dim=[0, 1, 2]).item()
        logits_values.append(pooled_value)
    logits_values = np.array(logits_values, dtype=np.float32).reshape(-1, 1)
    embeddings = embeddings.astype(np.float32)
    label = 1 if 'pos' in esmc_folder else 0
    labels = np.full((logits_values.shape[0],), label)
    return logits_values, embeddings, labels

train_pos_logits, train_pos_embeddings, train_pos_labels = load_esmc_features(esmc_folders['train_pos'])
train_neg_logits, train_neg_embeddings, train_neg_labels = load_esmc_features(esmc_folders['train_neg'])
test_pos_logits, test_pos_embeddings, test_pos_labels = load_esmc_features(esmc_folders['test_pos'])
test_neg_logits, test_neg_embeddings, test_neg_labels = load_esmc_features(esmc_folders['test_neg'])

train_logits = np.vstack((train_pos_logits, train_neg_logits))
train_embeddings = np.vstack((train_pos_embeddings, train_neg_embeddings))
train_labels = np.hstack((train_pos_labels, train_neg_labels))

test_logits = np.vstack((test_pos_logits, test_neg_logits))
test_embeddings = np.vstack((test_pos_embeddings, test_neg_embeddings))
test_labels = np.hstack((test_pos_labels, test_neg_labels))

print(f"训练集 logits 形状: {train_logits.shape}")
print(f"训练集 embeddings 形状: {train_embeddings.shape}")
print(f"训练集 labels 形状: {train_labels.shape}")
print(f"测试集 logits 形状: {test_logits.shape}")
print(f"测试集 embeddings 形状: {test_embeddings.shape}")
print(f"测试集 labels 形状: {test_labels.shape}")

def load_struct_features(json_path, sample_limit=5):
    with open(json_path, 'r') as f:
        json_data = json.load(f)
    data_list = []
    for idx, sample in enumerate(tqdm(json_data, desc=f'加载结构特征 from {json_path}')):
        required_keys = ['node_features', 'edge_features', 'label']
        if not all(key in sample for key in required_keys):
            print(f" 样本缺少必要的键: {sample}")
            continue
        node_features = torch.tensor(sample['node_features'], dtype=torch.float)
        edge_features = sample['edge_features']
        label = torch.tensor(sample['label'], dtype=torch.long)
        edges = torch.tensor(edge_features.get('edges', []), dtype=torch.long).t().contiguous()
        directions = torch.tensor(edge_features.get('directions', []), dtype=torch.float)
        rotations = torch.tensor(edge_features.get('rotations', []), dtype=torch.float).unsqueeze(1)
        edge_attr = torch.cat([directions, rotations], dim=1) if edges.size(0) > 0 else torch.empty((0, 4), dtype=torch.float)
        data = Data(x=node_features, edge_index=edges, edge_attr=edge_attr, y=label)
        data_list.append(data)
        if idx < sample_limit:
            print(f"样本 {idx+1}: 节点数量: {node_features.shape[0]}, 特征维度: {node_features.shape[1]}, 边数量: {edges.size(1)}")
    return data_list

train_struct_data = load_struct_features(struct_folders['train'])
test_struct_data = load_struct_features(struct_folders['test'])

def prepare_esm_only_data(struct_data, embeddings, logits):
    esm_only_data = []
    for i, data in enumerate(tqdm(struct_data, desc='准备仅 ESM-C 数据')):
        embedding = torch.tensor(embeddings[i], dtype=torch.float)  # [1152]
        logit = torch.tensor(logits[i], dtype=torch.float)  # [1], 移除 unsqueeze(0)
        combined_feature = torch.cat([embedding, logit], dim=0)  # [1153]
        num_nodes = data.x.shape[0]
        node_features = combined_feature.unsqueeze(0).repeat(num_nodes, 1)  # [num_nodes, 1153]
        edge_index = data.edge_index
        edge_attr = torch.zeros((edge_index.shape[1], 4), dtype=torch.float) if edge_index.shape[1] > 0 else torch.empty((0, 4), dtype=torch.float)
        new_data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=data.y)
        esm_only_data.append(new_data)
    return esm_only_data

train_esm_only_data = prepare_esm_only_data(train_struct_data, train_embeddings, train_logits)
test_esm_only_data = prepare_esm_only_data(test_struct_data, test_embeddings, test_logits)

class ProteinDataset(Dataset):
    def __init__(self, data_list):
        super(ProteinDataset, self).__init__()
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]

batch_size = 32
train_esm_only_dataset = ProteinDataset(train_esm_only_data)
test_esm_only_dataset = ProteinDataset(test_esm_only_data)
train_esm_only_loader = DataLoader(train_esm_only_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_esm_only_loader = DataLoader(test_esm_only_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

class DeepGATModel(nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim, hidden_dim, out_dim, num_heads=4, dropout=0.3, num_layers=3):
        super(DeepGATModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        self.edge_preprocess = nn.Sequential(
            nn.Linear(edge_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        for layer in range(num_layers):
            in_dim = node_feature_dim if layer == 0 else hidden_dim * num_heads
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=hidden_dim,
                heads=num_heads,
                dropout=dropout,
                edge_dim=hidden_dim,
                add_self_loops=True
            ))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim * num_heads))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim * num_heads, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        if edge_attr is not None:
            edge_attr = self.edge_preprocess(edge_attr)
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        if edge_attr is not None:
            edge_attr = self.edge_preprocess(edge_attr)
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GraphSAGEModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GraphSAGEModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        self.convs.append(SAGEConv(node_feature_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GINModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GINModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        for layer in range(num_layers):
            if layer == 0:
                nn_lin = nn.Sequential(nn.Linear(node_feature_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            else:
                nn_lin = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.convs.append(GINConv(nn_lin))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

# 三向交叉注意力 A学习BC
class CrossAttentionFusion(nn.Module):
    def __init__(self, feature_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.attn = nn.MultiheadAttention(
            embed_dim=feature_dim, 
            num_heads=num_heads, 
            dropout=dropout,
            # batch_first=False  # [L,B,E]
            batch_first=True  # [B,L,E]
        )
        self.norm1 = nn.LayerNorm(feature_dim)
        self.norm2 = nn.LayerNorm(feature_dim)
        self.norm3 = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)
    
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)  # 可学习权重
        self.fc = nn.Linear(feature_dim, 2)

    def forward(self, features_list):
        feat_gat, feat_sage, feat_gin = features_list
        self.attn_maps = {}

        def cross(q_src, kv_srcs, norm_layer, name):
            # 构造 QKV
            q = q_src.unsqueeze(1)              # [1, B, d]  改成了  [B,1,d]
            kv = torch.stack(kv_srcs, dim=1)    # [2, B, d] 改成了  [B,2,d]

            # 计算注意力
            attn_out, attn_weights = self.attn(q, kv, kv)  # attn_out: [B, 1, d], attn_weights: [B, 1, 2]
            attn_out = attn_out.squeeze(1)                 # [B, d]
            
            # 残差连接 + LayerNorm
            out = q_src + self.dropout(attn_out)
            out = norm_layer(out)

            # 保存注意力权重（平均所有样本）
            avg_attn = attn_weights.mean(dim=0).squeeze(0).detach().cpu().numpy()  # attn_weights 的形状是 [B, 1, 2]    mean(dim=0) 得到 [1, 2]，再 squeeze(0) 得 [2]
            self.attn_maps[name] = avg_attn  # 保存到模块属性

            return out

        # 三向交叉注意力
        out_gat  = cross(feat_gat,  [feat_sage, feat_gin], self.norm1, "GAT")
        out_sage = cross(feat_sage, [feat_gat, feat_gin], self.norm2, "SAGE")
        out_gin  = cross(feat_gin,  [feat_gat, feat_sage], self.norm3, "GIN")

        # 可学习权重融合
        weights = torch.softmax(self.fusion_weights, dim=0)
        fused = (out_gat * weights[0] + 
                out_sage * weights[1] + 
                out_gin * weights[2])

        return self.fc(fused)

def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path='best_model.pth'):
    best_test_acc = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler.step()
        train_acc, _, _ = test(model, train_loader, device)
        test_acc, _, _ = test(model, test_loader, device)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break
    model.load_state_dict(best_model_wts)
    return best_test_acc, best_model_wts

def test(model, loader, device):
    model.eval()
    correct = 0
    preds, trues = [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='评估'):
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(data.y.cpu().numpy())
            correct += (pred == data.y).sum().item()
    accuracy = correct / len(loader.dataset)
    return accuracy, trues, preds

def detailed_test(model, loader, device, models=None):
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='详细评估'):
            data = data.to(device)
            if isinstance(model, CrossAttentionFusion):
                if models is None:
                    raise ValueError("models dictionary required for CrossAttentionFusion evaluation")
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
                out = model(features_list)
            else:
                out = model(data)
            prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            pred = out.argmax(dim=1).cpu().numpy()
            true = data.y.cpu().numpy()
            preds.extend(pred)
            trues.extend(true)
            probs.extend(prob)
    acc = accuracy_score(trues, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(trues, preds, average='binary')
    mcc = matthews_corrcoef(trues, preds)
    auc = roc_auc_score(trues, probs)
    tn, fp, fn, tp = confusion_matrix(trues, preds).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics = {'acc': acc, 'mcc': mcc, 'auc': auc, 'sn': sn, 'sp': sp, 'precision': precision, 'recall': recall, 'f1': f1}
    return metrics

def optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=2):
    def objective(trial):
        hidden_dim = trial.suggest_int('hidden_dim', 64, 256)
        num_layers = trial.suggest_int('num_layers', 2, 5)
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
        if model_class == DeepGATModel:
            num_heads = trial.suggest_int('num_heads', 4, 8)
            model = DeepGATModel(
                node_feature_dim=model_params['node_feature_dim'],
                edge_feature_dim=model_params['edge_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_heads=num_heads,
                dropout=dropout,
                num_layers=num_layers
            ).to(device)
        elif model_class == GraphSAGEModel:
            model = GraphSAGEModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)
        elif model_class == GINModel:
            model = GINModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        best_acc, _ = train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10)
        return best_acc
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=2)
    print(f"{model_class.__name__} 最佳超参数: {study.best_params}")
    return study.best_params

def train_and_evaluate_models(train_loader, test_loader, device, output_folder):
    model_params = {"node_feature_dim": 1153, "edge_feature_dim": 4, "out_dim": 2}  # node_feature_dim 调整为 1153
    best_params = {}

    for model_class in [DeepGATModel, GraphSAGEModel, GINModel]:
        print(f"优化 {model_class.__name__}...")
        best_params[model_class.__name__] = optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=10)

    models = {
        "DeepGATModel": DeepGATModel(
            node_feature_dim=1153, edge_feature_dim=4, out_dim=2,
            hidden_dim=best_params["DeepGATModel"]["hidden_dim"],
            num_layers=best_params["DeepGATModel"]["num_layers"],
            dropout=best_params["DeepGATModel"]["dropout"],
            num_heads=best_params["DeepGATModel"]["num_heads"]
        ).to(device),
        "GraphSAGEModel": GraphSAGEModel(
            node_feature_dim=1153, out_dim=2,
            hidden_dim=best_params["GraphSAGEModel"]["hidden_dim"],
            num_layers=best_params["GraphSAGEModel"]["num_layers"],
            dropout=best_params["GraphSAGEModel"]["dropout"]
        ).to(device),
        "GINModel": GINModel(
            node_feature_dim=1153, out_dim=2,
            hidden_dim=best_params["GINModel"]["hidden_dim"],
            num_layers=best_params["GINModel"]["num_layers"],
            dropout=best_params["GINModel"]["dropout"]
        ).to(device)
    }

    results = {}
    for name, model in models.items():
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=best_params[name]["lr"], weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        save_path = os.path.join(output_folder, f"best_{name}_esm_only.pth")
        best_acc, _ = train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path=save_path)
        metrics = detailed_test(model, test_loader, device)
        results[name] = metrics
        print(f"{name} - Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}")

    print("\n### 三个模型性能对比 (ESM-C Only) ###")
    for name, metrics in results.items():
        print(f"{name}: Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}, "
              f"Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, "
              f"Sn: {metrics['sn']:.4f}, Sp: {metrics['sp']:.4f}")

    for name, model in models.items():
        model.load_state_dict(torch.load(os.path.join(output_folder, f"best_{name}_esm_only.pth")))
        model.eval()

    return results, models

def train_cross_attention_fusion(models, train_loader, test_loader, device, output_folder, num_epochs=50, patience=10):
    fusion_module = CrossAttentionFusion(feature_dim=256, num_heads=4, dropout=0.1).to(device)
    optimizer_fusion = torch.optim.Adam(fusion_module.parameters(), lr=0.001, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler_fusion = torch.optim.lr_scheduler.StepLR(optimizer_fusion, step_size=10, gamma=0.1)

    print("\n### 训练交叉注意力融合模块 (ESM-C Only) ###")
    best_fusion_acc = 0
    best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        fusion_module.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'融合训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            with torch.no_grad():
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
            out = fusion_module(features_list)
            loss = criterion(out, data.y)
            optimizer_fusion.zero_grad()
            loss.backward()
            optimizer_fusion.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler_fusion.step()

        fusion_module.eval()
        preds, trues = [], []
        with torch.no_grad():
            for data in test_loader:
                data = data.to(device)
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
                out = fusion_module(features_list)
                pred = out.argmax(dim=1)
                preds.extend(pred.cpu().numpy())
                trues.extend(data.y.cpu().numpy())
        test_acc = accuracy_score(trues, preds)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Fusion Test Acc: {test_acc:.4f}")

        if test_acc > best_fusion_acc:
            best_fusion_acc = test_acc
            best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
            epochs_no_improve = 0
            torch.save(fusion_module.state_dict(), os.path.join(output_folder, "best_cross_attention_fusion_esm_only.pth"))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break

    fusion_module.load_state_dict(best_fusion_wts)
    fusion_metrics = detailed_test(fusion_module, test_loader, device, models=models)
    return fusion_module, fusion_metrics

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    log_path = setup_jupyter_logger(aggregated_output_folder, log_prefix="AFP_train")
    print(f" 日志记录开始，输出文件：{log_path}")

    print("\n### 消融实验: 仅使用 ESM-C 特征 ###")
    results_esm, models_esm = train_and_evaluate_models(train_esm_only_loader, test_esm_only_loader, device, aggregated_output_folder)
    fusion_module_esm, fusion_metrics_esm = train_cross_attention_fusion(models_esm, train_esm_only_loader, test_esm_only_loader, device, aggregated_output_folder)
    results_esm["CrossAttentionFusion"] = fusion_metrics_esm
    print(f"CrossAttentionFusion (ESM-C Only) - Acc: {fusion_metrics_esm['acc']:.4f}, MCC: {fusion_metrics_esm['mcc']:.4f}, AUC: {fusion_metrics_esm['auc']:.4f}, "
          f"Precision: {fusion_metrics_esm['precision']:.4f}, Recall: {fusion_metrics_esm['recall']:.4f}, F1: {fusion_metrics_esm['f1']:.4f}, "
          f"Sn: {fusion_metrics_esm['sn']:.4f}, Sp: {fusion_metrics_esm['sp']:.4f}")

    print("=" * 80)
    print(f" 实验全部完成！日志文件保存路径: {log_path}")
    print("=" * 80)

### 仅使用结构特征

In [8]:
import os
import numpy as np
import json
from tqdm import tqdm
import random
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import (
    GATConv, SAGEConv, GINConv, global_mean_pool
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, matthews_corrcoef,
    roc_auc_score, confusion_matrix
)
import copy
import optuna

# 文件路径
# struct_folders = {
#     'train': '/content/drive/MyDrive/AFP_work/pdb_features/aggregated/train_dataset.json',
#     'test': '/content/drive/MyDrive/AFP_work/pdb_features/aggregated/test_dataset.json'
# }

# aggregated_output_folder = '/content/drive/MyDrive/AFP_work/esmc_struct_aggregated'
# os.makedirs(aggregated_output_folder, exist_ok=True)

struct_folders = {
    'train': 'pdb_features_9A/aggregated/train_dataset.json',
    'test': 'pdb_features_9A/aggregated/test_dataset.json'
}

aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention/Ablation experiments /only_af2'
os.makedirs(aggregated_output_folder, exist_ok=True)

# 数据加载和预处理
def load_struct_features(json_path, sample_limit=5):
    with open(json_path, 'r') as f:
        json_data = json.load(f)
    data_list = []
    for idx, sample in enumerate(tqdm(json_data, desc=f'加载结构特征 from {json_path}')):
        required_keys = ['node_features', 'edge_features', 'label']
        if not all(key in sample for key in required_keys):
            print(f"  样本缺少必要的键: {sample}")
            continue
        node_features = torch.tensor(sample['node_features'], dtype=torch.float)
        edge_features = sample['edge_features']
        label = torch.tensor(sample['label'], dtype=torch.long)
        edges = torch.tensor(edge_features.get('edges', []), dtype=torch.long).t().contiguous()
        directions = torch.tensor(edge_features.get('directions', []), dtype=torch.float)
        rotations = torch.tensor(edge_features.get('rotations', []), dtype=torch.float).unsqueeze(1)
        edge_attr = torch.cat([directions, rotations], dim=1) if edges.size(0) > 0 else torch.empty((0, 4), dtype=torch.float)
        data = Data(x=node_features, edge_index=edges, edge_attr=edge_attr, y=label)
        data_list.append(data)
        if idx < sample_limit:
            print(f"样本 {idx+1}: 节点数量: {node_features.shape[0]}, 特征维度: {node_features.shape[1]}, 边数量: {edges.size(1)}")
    return data_list

train_struct_data = load_struct_features(struct_folders['train'])
test_struct_data = load_struct_features(struct_folders['test'])

def normalize_features(train_data_list, test_data_list=None):
    node_scaler = StandardScaler()
    edge_scaler = StandardScaler()
    all_node_features = np.vstack([data.x.numpy()[:, :3] for data in train_data_list])  # 只取结构特征
    all_edge_features = np.vstack([data.edge_attr.numpy() for data in train_data_list if data.edge_attr.size(0) > 0])
    node_scaler.fit(all_node_features)
    if all_edge_features.size > 0:
        edge_scaler.fit(all_edge_features)
    for data in train_data_list:
        data.x = torch.tensor(node_scaler.transform(data.x.numpy()[:, :3]), dtype=torch.float)
        if data.edge_attr.size(0) > 0:
            data.edge_attr = torch.tensor(edge_scaler.transform(data.edge_attr.numpy()), dtype=torch.float)
    if test_data_list:
        for data in test_data_list:
            data.x = torch.tensor(node_scaler.transform(data.x.numpy()[:, :3]), dtype=torch.float)
            if data.edge_attr.size(0) > 0:
                data.edge_attr = torch.tensor(edge_scaler.transform(data.edge_attr.numpy()), dtype=torch.float)
    return train_data_list, test_data_list

train_struct_data, test_struct_data = normalize_features(train_struct_data, test_struct_data)

class ProteinDataset(Dataset):
    def __init__(self, data_list):
        super(ProteinDataset, self).__init__()
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]

batch_size = 32
train_struct_only_dataset = ProteinDataset(train_struct_data)
test_struct_only_dataset = ProteinDataset(test_struct_data)
train_struct_only_loader = DataLoader(train_struct_only_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_struct_only_loader = DataLoader(test_struct_only_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# 模型定义
class DeepGATModel(nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim, hidden_dim, out_dim, num_heads=4, dropout=0.3, num_layers=3):
        super(DeepGATModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        self.edge_preprocess = nn.Sequential(
            nn.Linear(edge_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        for layer in range(num_layers):
            in_dim = node_feature_dim if layer == 0 else hidden_dim * num_heads
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=hidden_dim,
                heads=num_heads,
                dropout=dropout,
                edge_dim=hidden_dim,
                add_self_loops=True
            ))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim * num_heads))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim * num_heads, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data, print_shapes=False):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        # 仅在第一次批次打印形状（每轮试验一次）
        if print_shapes:
            print(f"输入 - 节点特征形状: {x.shape}, 边特征形状: {edge_attr.shape if edge_attr is not None else '无'}, 边索引形状: {edge_index.shape}")

        # 预处理边特征
        if edge_attr is not None and edge_attr.shape[0] > 0:
            edge_attr = self.edge_preprocess(edge_attr)
            if print_shapes:
                print(f"预处理后边特征形状: {edge_attr.shape}")
        else:
            edge_attr = None
            if print_shapes:
                print("无边特征，使用默认边处理")

        # 逐层处理
        for i, (conv, bn) in enumerate(zip(self.convs, self.batch_norms)):
            x = conv(x, edge_index, edge_attr)
            if print_shapes:
                print(f"GATConv层 {i+1} 输出形状: {x.shape}")
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)

        # 池化
        x = self.readout(x, batch)
        if print_shapes:
            print(f"池化后特征形状: {x.shape}")

        # 全连接层
        x = self.fc1(x)
        if print_shapes:
            print(f"FC1输出形状: {x.shape}")
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        if print_shapes:
            print(f"FC2输出形状: {x.shape}")  # 应为 [batch_size, 2]

        return x

    def get_last_layer_features(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        if edge_attr is not None:
            edge_attr = self.edge_preprocess(edge_attr)
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GraphSAGEModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GraphSAGEModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        self.convs.append(SAGEConv(node_feature_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GINModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GINModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        for layer in range(num_layers):
            if layer == 0:
                nn_lin = nn.Sequential(nn.Linear(node_feature_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            else:
                nn_lin = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.convs.append(GINConv(nn_lin))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

# class CrossAttentionFusion(nn.Module):
#     def __init__(self, feature_dim, num_heads=4, dropout=0.1):
#         super(CrossAttentionFusion, self).__init__()
#         self.attention = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=num_heads, dropout=dropout)
#         self.norm = nn.LayerNorm(feature_dim)
#         self.fc = nn.Linear(feature_dim, 2)

#     def forward(self, features_list):
#         feats = torch.stack(features_list, dim=0)
#         attn_output, _ = self.attention(feats, feats, feats)
#         fused_feats = attn_output.mean(dim=0)
#         fused_feats = self.norm(fused_feats)
#         out = self.fc(fused_feats)
#         return out


# 三向交叉注意力 A学习BC
class CrossAttentionFusion(nn.Module):
    def __init__(self, feature_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.attn = nn.MultiheadAttention(
            embed_dim=feature_dim, 
            num_heads=num_heads, 
            dropout=dropout,
            # batch_first=False  # [L,B,E]
            batch_first=True  # [B,L,E]
        )
        self.norm1 = nn.LayerNorm(feature_dim)
        self.norm2 = nn.LayerNorm(feature_dim)
        self.norm3 = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)
        
        # 更灵活的融合方式
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)  # 可学习权重
        self.fc = nn.Linear(feature_dim, 2)

    def forward(self, features_list):
        feat_gat, feat_sage, feat_gin = features_list
        self.attn_maps = {}

        def cross(q_src, kv_srcs, norm_layer, name):
            # 构造 QKV
            q = q_src.unsqueeze(1)              # [1, B, d]  改成了  [B,1,d]
            kv = torch.stack(kv_srcs, dim=1)    # [2, B, d] 改成了  [B,2,d]

            # 计算注意力
            attn_out, attn_weights = self.attn(q, kv, kv)  # attn_out: [B, 1, d], attn_weights: [B, 1, 2]
            attn_out = attn_out.squeeze(1)                 # [B, d]
            
            # 残差连接 + LayerNorm
            out = q_src + self.dropout(attn_out)
            out = norm_layer(out)

            # 保存注意力权重（平均所有样本）
            avg_attn = attn_weights.mean(dim=0).squeeze(0).detach().cpu().numpy()  # attn_weights 的形状是 [B, 1, 2]    mean(dim=0) 得到 [1, 2]，再 squeeze(0) 得 [2]
            self.attn_maps[name] = avg_attn  # 保存到模块属性

            return out

        # 三向交叉注意力
        out_gat  = cross(feat_gat,  [feat_sage, feat_gin], self.norm1, "GAT")
        out_sage = cross(feat_sage, [feat_gat, feat_gin], self.norm2, "SAGE")
        out_gin  = cross(feat_gin,  [feat_gat, feat_sage], self.norm3, "GIN")

        # 可学习权重融合（替代简单的mean）
        weights = torch.softmax(self.fusion_weights, dim=0)
        fused = (out_gat * weights[0] + 
                out_sage * weights[1] + 
                out_gin * weights[2])

        return self.fc(fused)

def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path='best_model.pth'):
    best_test_acc = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler.step()
        train_acc, _, _ = test(model, train_loader, device)
        test_acc, _, _ = test(model, test_loader, device)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break
    model.load_state_dict(best_model_wts)
    return best_test_acc, best_model_wts

def test(model, loader, device):
    model.eval()
    correct = 0
    preds, trues = [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='评估'):
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(data.y.cpu().numpy())
            correct += (pred == data.y).sum().item()
    accuracy = correct / len(loader.dataset)
    return accuracy, trues, preds

def detailed_test(model, loader, device, models=None):
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='详细评估'):
            data = data.to(device)
            if isinstance(model, CrossAttentionFusion):
                if models is None:
                    raise ValueError("models dictionary required for CrossAttentionFusion evaluation")
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
                out = model(features_list)
            else:
                out = model(data)
            prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            pred = out.argmax(dim=1).cpu().numpy()
            true = data.y.cpu().numpy()
            preds.extend(pred)
            trues.extend(true)
            probs.extend(prob)
    acc = accuracy_score(trues, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(trues, preds, average='binary')
    mcc = matthews_corrcoef(trues, preds)
    auc = roc_auc_score(trues, probs)
    tn, fp, fn, tp = confusion_matrix(trues, preds).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics = {'acc': acc, 'mcc': mcc, 'auc': auc, 'sn': sn, 'sp': sp, 'precision': precision, 'recall': recall, 'f1': f1}
    return metrics

def optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=5):
    def objective(trial):
        hidden_dim = trial.suggest_int('hidden_dim', 64, 512)
        num_layers = trial.suggest_int('num_layers', 2, 6)
        um_heads = trial.suggest_int('num_heads', 2, 16)
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)
        if model_class == DeepGATModel:
            num_heads = trial.suggest_int('num_heads', 4, 8)
            model = DeepGATModel(
                node_feature_dim=model_params['node_feature_dim'],
                edge_feature_dim=model_params['edge_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_heads=num_heads,
                dropout=dropout,
                num_layers=num_layers
            ).to(device)
        elif model_class == GraphSAGEModel:
            model = GraphSAGEModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)
        elif model_class == GINModel:
            model = GINModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        best_acc, _ = train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10)
        return best_acc
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=5)
    print(f"{model_class.__name__} 最佳超参数: {study.best_params}")
    return study.best_params

def train_and_evaluate_models(train_loader, test_loader, device, output_folder):
    model_params = {"node_feature_dim": 3, "edge_feature_dim": 4, "out_dim": 2}  # node_feature_dim 调整为 3
    best_params = {}

    for model_class in [DeepGATModel, GraphSAGEModel, GINModel]:
        print(f"优化 {model_class.__name__}...")
        best_params[model_class.__name__] = optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=10)

    models = {
        "DeepGATModel": DeepGATModel(
            node_feature_dim=3, edge_feature_dim=4, out_dim=2,
            hidden_dim=best_params["DeepGATModel"]["hidden_dim"],
            num_layers=best_params["DeepGATModel"]["num_layers"],
            dropout=best_params["DeepGATModel"]["dropout"],
            num_heads=best_params["DeepGATModel"]["num_heads"]
        ).to(device),
        "GraphSAGEModel": GraphSAGEModel(
            node_feature_dim=3, out_dim=2,
            hidden_dim=best_params["GraphSAGEModel"]["hidden_dim"],
            num_layers=best_params["GraphSAGEModel"]["num_layers"],
            dropout=best_params["GraphSAGEModel"]["dropout"]
        ).to(device),
        "GINModel": GINModel(
            node_feature_dim=3, out_dim=2,
            hidden_dim=best_params["GINModel"]["hidden_dim"],
            num_layers=best_params["GINModel"]["num_layers"],
            dropout=best_params["GINModel"]["dropout"]
        ).to(device)
    }

    results = {}
    for name, model in models.items():
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=best_params[name]["lr"], weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        save_path = os.path.join(output_folder, f"best_{name}_struct_only.pth")
        best_acc, _ = train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path=save_path)
        metrics = detailed_test(model, test_loader, device)
        results[name] = metrics
        print(f"{name} - Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}")

    print("\n### 三个模型性能对比 (Struct Only) ###")
    for name, metrics in results.items():
        print(f"{name}: Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}, "
              f"Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, "
              f"Sn: {metrics['sn']:.4f}, Sp: {metrics['sp']:.4f}")

    for name, model in models.items():
        model.load_state_dict(torch.load(os.path.join(output_folder, f"best_{name}_struct_only.pth")))
        model.eval()

    return results, models

def train_cross_attention_fusion(models, train_loader, test_loader, device, output_folder, num_epochs=50, patience=10):
    fusion_module = CrossAttentionFusion(feature_dim=256, num_heads=4, dropout=0.1).to(device)
    optimizer_fusion = torch.optim.Adam(fusion_module.parameters(), lr=0.001, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler_fusion = torch.optim.lr_scheduler.StepLR(optimizer_fusion, step_size=10, gamma=0.1)

    print("\n### 训练交叉注意力融合模块 (Struct Only) ###")
    best_fusion_acc = 0
    best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        fusion_module.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'融合训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            with torch.no_grad():
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
            out = fusion_module(features_list)
            loss = criterion(out, data.y)
            optimizer_fusion.zero_grad()
            loss.backward()
            optimizer_fusion.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler_fusion.step()

        fusion_module.eval()
        preds, trues = [], []
        with torch.no_grad():
            for data in test_loader:
                data = data.to(device)
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
                out = fusion_module(features_list)
                pred = out.argmax(dim=1)
                preds.extend(pred.cpu().numpy())
                trues.extend(data.y.cpu().numpy())
        test_acc = accuracy_score(trues, preds)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Fusion Test Acc: {test_acc:.4f}")

        if test_acc > best_fusion_acc:
            best_fusion_acc = test_acc
            best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
            epochs_no_improve = 0
            torch.save(fusion_module.state_dict(), os.path.join(output_folder, "best_cross_attention_fusion_struct_only.pth"))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break

    fusion_module.load_state_dict(best_fusion_wts)
    fusion_metrics = detailed_test(fusion_module, test_loader, device, models=models)
    return fusion_module, fusion_metrics

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 初始化日志系统
    log_path = setup_jupyter_logger(aggregated_output_folder, log_prefix="AFP_train")
    print(f" 日志记录开始，输出文件：{log_path}")

    print("\n### 消融实验: 仅使用结构特征 ###")
    results_struct, models_struct = train_and_evaluate_models(train_struct_only_loader, test_struct_only_loader, device, aggregated_output_folder)
    fusion_module_struct, fusion_metrics_struct = train_cross_attention_fusion(models_struct, train_struct_only_loader, test_struct_only_loader, device, aggregated_output_folder)
    results_struct["CrossAttentionFusion"] = fusion_metrics_struct
    print(f"CrossAttentionFusion (Struct Only) - Acc: {fusion_metrics_struct['acc']:.4f}, MCC: {fusion_metrics_struct['mcc']:.4f}, AUC: {fusion_metrics_struct['auc']:.4f}, "
          f"Precision: {fusion_metrics_struct['precision']:.4f}, Recall: {fusion_metrics_struct['recall']:.4f}, F1: {fusion_metrics_struct['f1']:.4f}, "
          f"Sn: {fusion_metrics_struct['sn']:.4f}, Sp: {fusion_metrics_struct['sp']:.4f}")


    print("=" * 80)
    print(f" 实验全部完成！日志文件保存路径: {log_path}")
    print("=" * 80)

#  Test

In [ ]:
import os
import numpy as np
import pandas as pd
import json
from tqdm import tqdm
import random
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader

from torch_geometric.nn import (
    GATConv, SAGEConv, GINConv, Set2Set, global_mean_pool
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, precision_recall_fscore_support,
    matthews_corrcoef, roc_auc_score, confusion_matrix
)
import copy
import optuna
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
from sklearn.manifold import TSNE
import seaborn as sns
from torch_geometric.loader import DataLoader


# esmc_folders = {
#     'train_pos': 'esmc_600_train_pos',
#     'train_neg': 'esmc_600_train_neg',
#     'test_pos': 'esmc_600_test_pos',
#     'test_neg': 'esmc_600_test_neg'
# }

# struct_folders = {
#     'train': 'pdb_features_9A/aggregated/train_dataset.json',
#     'test': 'pdb_features_9A/aggregated/test_dataset.json'
# }

# # aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention_new_heatmap'
# aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention_251122'
# # aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention'


# esmc_folders = {
#     'train_pos': 'esmc_600_train_pos',
#     'train_neg': 'esmc_600_train_neg',
#     'test_pos': 'test_data_low_plddt/esmc-600/test_pos_low_plddt',
#     'test_neg': 'test_data_low_plddt/esmc-600/test_neg_low_plddt'
# }

# struct_folders = {
#     'train': 'pdb_features_9A/aggregated/train_dataset.json',
#     'test': 'test_data_low_plddt/pdb_features_9A/aggregated/test_dataset.json'
# }



esmc_folders = {
    'test_pos': 'external_set_result/esmc_600_external_test/esmc_600_external_pos',
    'test_neg': 'external_set_result/esmc_600_external_test/esmc_600_external_neg'
}

struct_folders = {
    'test': 'external_set_result/pdb_features_9A/aggregated/test_dataset.json'
}


# aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention_new_heatmap'
# aggregated_output_folder = 'test_data_low_plddt/result_251123'
# aggregated_output_folder = 'esmc_struct_aggregated_9A_CrossAttention'

aggregated_output_folder = 'external_set_result/result_251124'


os.makedirs(aggregated_output_folder, exist_ok=True)

#***************************1、加载 ESM-C 特征***************************
def load_esmc_features(esmc_folder):
    logits_path = os.path.join(esmc_folder, 'combined_logits.npy')
    embeddings_path = os.path.join(esmc_folder, 'combined_embeddings.npy')
    logits = np.load(logits_path, allow_pickle=True)
    embeddings = np.load(embeddings_path, allow_pickle=True)

    print(f"Logits[0] 类型: {type(logits[0])}, 值: {logits[0]}")  #  类型 <class 'numpy.ndarray'>
    print("Logits sample:", logits[0])
    print("Embeddings sample:", embeddings[0])

    logits_values = []
    for l in logits:
        forward_data = l[0] if isinstance(l, np.ndarray) else l
        sequence_tensor = forward_data.sequence
        sequence_tensor = sequence_tensor.to(device='cpu', dtype=torch.float32)
        pooled_value = sequence_tensor.mean(dim=[0, 1, 2]).item()
        logits_values.append(pooled_value)

    logits_values = np.array(logits_values, dtype=np.float32).reshape(-1, 1)
    embeddings = embeddings.astype(np.float32)
    label = 1 if 'pos' in esmc_folder else 0
    labels = np.full((logits_values.shape[0],), label)
    return logits_values, embeddings, labels
# 加载训练集和测试集的 ESM-C 特征
train_pos_logits, train_pos_embeddings, train_pos_labels = load_esmc_features(esmc_folders['train_pos'])
train_neg_logits, train_neg_embeddings, train_neg_labels = load_esmc_features(esmc_folders['train_neg'])
test_pos_logits, test_pos_embeddings, test_pos_labels = load_esmc_features(esmc_folders['test_pos'])
test_neg_logits, test_neg_embeddings, test_neg_labels = load_esmc_features(esmc_folders['test_neg'])
# 合并训练集和测试集特征
train_logits = np.vstack((train_pos_logits, train_neg_logits))
train_embeddings = np.vstack((train_pos_embeddings, train_neg_embeddings))
train_labels = np.hstack((train_pos_labels, train_neg_labels))

test_logits = np.vstack((test_pos_logits, test_neg_logits))
test_embeddings = np.vstack((test_pos_embeddings, test_neg_embeddings))
test_labels = np.hstack((test_pos_labels, test_neg_labels))

print(f"训练集 logits 形状: {train_logits.shape}")  # （2400,1）
print(f"训练集 embeddings 形状: {train_embeddings.shape}") # （2400,1152）
print(f"训练集 labels 形状: {train_labels.shape}") # （2400,）

print(f"测试集 logits 形状: {test_logits.shape}") # （616,1）
print(f"测试集 embeddings 形状: {test_embeddings.shape}")  # （616,1152）
print(f"测试集 labels 形状: {test_labels.shape}") # （616,）


#***************************2、加载结构特征***************************
def load_struct_features(json_path, sample_limit=5):
    with open(json_path, 'r') as f:
        json_data = json.load(f)
    data_list = []
    for idx, sample in enumerate(tqdm(json_data, desc=f'加载结构特征 from {json_path}')):
        required_keys = ['node_features', 'edge_features', 'label']
        if not all(key in sample for key in required_keys):
            print(f"[ERROR] 样本缺少必要的键: {sample}")
            continue
        node_features = sample['node_features']
        edge_features = sample['edge_features']
        label = sample['label']
        edges = edge_features.get('edges', [])
        directions = edge_features.get('directions', [])
        rotations = edge_features.get('rotations', [])
        num_edges = len(edges)
        if not (len(directions) == num_edges and len(rotations) == num_edges):
            print(f"[ERROR] 边的数量与方向或旋转数量不匹配: {sample}")
            continue
        if num_edges > 0:
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            directions = torch.tensor(directions, dtype=torch.float)
            rotations = torch.tensor(rotations, dtype=torch.float).unsqueeze(1)
            edge_attr = torch.cat([directions, rotations], dim=1)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, 4), dtype=torch.float)
        node_features = torch.tensor(node_features, dtype=torch.float)
        label = torch.tensor(label, dtype=torch.long)
        data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=label)
        data_list.append(data)
        if idx < sample_limit:
            num_nodes = node_features.shape[0]
            node_feature_dim = node_features.shape[1]
            print(f"样本 {idx+1}: 节点数量: {num_nodes}, 节点特征维度: {node_feature_dim}, 边数量: {num_edges}")
            if num_edges > 0:
                print(f"  边特征维度: {edge_attr.shape[1]}")
            print("-" * 50)
    unique_node_feature_dims = set([data.x.shape[1] for data in data_list])
    unique_edge_feature_dims = set([data.edge_attr.shape[1] for data in data_list if data.edge_attr.shape[0] > 0])
    print(f"\n所有样本中唯一的节点特征维度: {unique_node_feature_dims}")  # 3
    print(f"所有样本中唯一的边特征维度: {unique_edge_feature_dims}")  # 4
    return data_list

train_struct_data = load_struct_features(struct_folders['train'])
test_struct_data = load_struct_features(struct_folders['test'])

for i in range(min(3, len(train_struct_data))):
    data = train_struct_data[i]
    print(f"样本 {i+1} - 节点特征: {data.x.shape}, 边特征: {data.edge_attr.shape}")

##*************************** 特征标准化 ***************************
def normalize_features(train_data_list, test_data_list=None):
    node_scaler = StandardScaler()
    edge_scaler = StandardScaler()
    all_node_features = np.concatenate([data.x.numpy() for data in train_data_list], axis=0)
    all_edge_features = np.concatenate([data.edge_attr.numpy() for data in train_data_list if data.edge_attr.shape[0] > 0], axis=0)
    node_scaler.fit(all_node_features)
    if all_edge_features.size > 0:
        edge_scaler.fit(all_edge_features)
    for data in train_data_list:
        data.x = torch.tensor(node_scaler.transform(data.x.numpy()), dtype=torch.float)
        if data.edge_attr.shape[0] > 0:
            data.edge_attr = torch.tensor(edge_scaler.transform(data.edge_attr.numpy()), dtype=torch.float)
    if test_data_list:
        for data in test_data_list:
            data.x = torch.tensor(node_scaler.transform(data.x.numpy()), dtype=torch.float)
            if data.edge_attr.shape[0] > 0:
                data.edge_attr = torch.tensor(edge_scaler.transform(data.edge_attr.numpy()), dtype=torch.float)
    return train_data_list, test_data_list

train_struct_data, test_struct_data = normalize_features(train_struct_data, test_struct_data)

# #*************************** 整合 ESM-C 特征 ***************************
def integrate_features(data_list, embeddings, logits):
    if len(data_list) != len(embeddings) or len(data_list) != len(logits):
        raise ValueError(f"data_list, embeddings 和 logits 长度不匹配: {len(data_list)} vs {len(embeddings)} vs {len(logits)}")
    for i, data in enumerate(tqdm(data_list, desc='整合 ESM-C embeddings 和 logits')):
        embedding = torch.tensor(embeddings[i], dtype=torch.float)  # [1152]
        logit = torch.tensor(logits[i], dtype=torch.float).squeeze()  # [1] -> 标量
        combined_feature = torch.cat([embedding, logit.unsqueeze(0)], dim=0)  # [1152 + 1 = 1153]
        num_nodes = data.x.shape[0]
        combined_expanded = combined_feature.unsqueeze(0).repeat(num_nodes, 1)  # [num_nodes, 1153]
        data.x = torch.cat([data.x, combined_expanded], dim=1)  # [num_nodes, 3 + 1153 = 1156]
    return data_list

train_struct_data = integrate_features(train_struct_data, train_embeddings, train_logits)
test_struct_data = integrate_features(test_struct_data, test_embeddings, test_logits)

print(f"训练集第一个样本的节点特征维度（整合后）: {train_struct_data[0].x.shape[1]}")  # 1156
print(f"测试集第一个样本的节点特征维度（整合后）: {test_struct_data[0].x.shape[1]}") # 1156

# #***************************创建数据集和数据加载器**#***************************
class ProteinDataset(Dataset):
    def __init__(self, data_list):
        super(ProteinDataset, self).__init__()
        self.data_list = data_list
    def len(self):
        return len(self.data_list)
    def get(self, idx):
        return self.data_list[idx]

train_dataset = ProteinDataset(train_struct_data)
test_dataset = ProteinDataset(test_struct_data)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef, roc_auc_score, confusion_matrix


class DeepGATModel(nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim, hidden_dim, out_dim, num_heads=4, dropout=0.3, num_layers=3):
        super(DeepGATModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)

        # 边特征预处理层
        self.edge_preprocess = nn.Sequential(
            nn.Linear(edge_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # 堆叠 GAT 层
        for layer in range(num_layers):
            in_dim = node_feature_dim if layer == 0 else hidden_dim * num_heads
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=hidden_dim,
                heads=num_heads,
                dropout=dropout,
                edge_dim=hidden_dim,  # 调整为预处理后的边特征维度
                add_self_loops=True  # 添加自环，增强稳定性。能提升稳定性（每个节点至少保留自身信息）
            ))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim * num_heads))

        # 替换 Set2Set 为更简单的池化
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim * num_heads, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        if edge_attr is not None:
            edge_attr = self.edge_preprocess(edge_attr)

        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)

        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        if edge_attr is not None:
            edge_attr = self.edge_preprocess(edge_attr)

        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            x = self.dropout(x)

        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GraphSAGEModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GraphSAGEModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        self.convs.append(SAGEConv(node_feature_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x

class GINModel(nn.Module):
    def __init__(self, node_feature_dim, hidden_dim, out_dim, num_layers=3, dropout=0.5):
        super(GINModel, self).__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout)
        for layer in range(num_layers):
            if layer == 0:
                nn_lin = nn.Sequential(nn.Linear(node_feature_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            else:
                nn_lin = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
            self.convs.append(GINConv(nn_lin))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
        self.readout = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.fc2 = nn.Linear(256, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def get_last_layer_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = self.dropout(x)
        x = self.readout(x, batch)
        x = self.fc1(x)
        return x
 
# 三向交叉注意力 A学习BC
class CrossAttentionFusion(nn.Module):
    def __init__(self, feature_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.attn = nn.MultiheadAttention(
            embed_dim=feature_dim, 
            num_heads=num_heads, 
            dropout=dropout,
            # batch_first=False  # [L,B,E]
            batch_first=True  # [B,L,E]
        )
        self.norm1 = nn.LayerNorm(feature_dim)
        self.norm2 = nn.LayerNorm(feature_dim)
        self.norm3 = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)

        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)  # 可学习权重
        self.fc = nn.Linear(feature_dim, 2)

    def forward(self, features_list):
        feat_gat, feat_sage, feat_gin = features_list
        self.attn_maps = {}

        def cross(q_src, kv_srcs, norm_layer, name):
            # 构造 QKV
            q = q_src.unsqueeze(1)              # [1, B, d]  改成了  [B,1,d]
            kv = torch.stack(kv_srcs, dim=1)    # [2, B, d] 改成了  [B,2,d]

            # 计算注意力
            attn_out, attn_weights = self.attn(q, kv, kv)  # attn_out: [B, 1, d], attn_weights: [B, 1, 2]
            attn_out = attn_out.squeeze(1)                 # [B, d]
            
            # 残差连接 + LayerNorm
            out = q_src + self.dropout(attn_out)
            out = norm_layer(out)

            # 保存注意力权重（平均所有样本）
            avg_attn = attn_weights.mean(dim=0).squeeze(0).detach().cpu().numpy()  # attn_weights 的形状是 [B, 1, 2]    mean(dim=0) 得到 [1, 2]，再 squeeze(0) 得 [2]
            self.attn_maps[name] = avg_attn  # 保存到模块属性

            return out

        # 三向交叉注意力
        out_gat  = cross(feat_gat,  [feat_sage, feat_gin], self.norm1, "GAT")
        out_sage = cross(feat_sage, [feat_gat, feat_gin], self.norm2, "SAGE")
        out_gin  = cross(feat_gin,  [feat_gat, feat_sage], self.norm3, "GIN")

        # 可学习权重融合
        weights = torch.softmax(self.fusion_weights, dim=0)
        fused = (out_gat * weights[0] + 
                out_sage * weights[1] + 
                out_gin * weights[2])

        return self.fc(fused)

# 3x3 注意力权重图
def plot_attention_matrix(fusion_module, output_folder, epoch, mode="cross"):
    os.makedirs(output_folder, exist_ok=True)
    weights = np.zeros((3, 3))
    gat = fusion_module.attn_maps.get("GAT", [0, 0])
    sage = fusion_module.attn_maps.get("SAGE", [0, 0])
    gin = fusion_module.attn_maps.get("GIN", [0, 0])
    weights[0, 1:] = gat
    weights[1, [0, 2]] = sage
    weights[2, :2] = gin

    # # 是否保留自注意力
    # if mode == "self":
    #     diag_vals = torch.softmax(fusion_module.fusion_weights, dim=0).detach().cpu().numpy()
    #     np.fill_diagonal(weights, diag_vals)

    # 设置对角线为空白   下面sns.heatmap 更改为annot=annot
    annot = weights.copy().astype(object)

    for i in range(3):
        for j in range(3):
            if i == j:
                annot[i][j] = ""        # 对角线留空
            else:
                annot[i][j] = f"{weights[i][j]:.2f}"  # 数字格式化为两位小数

    if mode == "self":
        diag_vals = torch.softmax(fusion_module.fusion_weights, dim=0).detach().cpu().numpy()
        for i in range(3):
            annot[i][i] = f"{diag_vals[i]:.2f}"

    sns.heatmap(weights, annot=annot, fmt="", cmap="YlGnBu",
                xticklabels=["GAT", "SAGE", "GIN"],
                yticklabels=["GAT", "SAGE", "GIN"],
                vmin=0, vmax=1)
    plt.title(f"Attention Map (Epoch {epoch}, mode={mode})")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f"attn_epoch_{epoch}_{mode}.png"), dpi=600, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# CKA 相似性分析
def linear_cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    X: [N, d1]
    Y: [N, d2]
    返回一个标量 CKA 值，用于衡量两个模型特征空间的相似度
    """
    # 中心化
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)
    # Gram 矩阵
    Gx = X.T @ X
    Gy = Y.T @ Y
    numerator = torch.norm(X.T @ Y, p='fro') ** 2
    denom = torch.norm(Gx, p='fro') * torch.norm(Gy, p='fro')

    return (numerator / denom).item()

# CKA 热图
def plot_cka_heatmap(cka_matrix, output_folder):
    labels = ["GAT", "SAGE", "GIN"]

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cka_matrix,
        annot=True,
        fmt=".3f",
        cmap="YlGnBu",
        xticklabels=labels,
        yticklabels=labels,
        vmin=0,
        vmax=1
    )
    plt.title("CKA Similarity Matrix of GNN Branches", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "cka_heatmap.png"), dpi=600)
    plt.close()

def collect_embeddings(model, loader, device):
    model.eval()
    all_features = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            feat = model.get_last_layer_features(data)   # shape [batch, 256]
            all_features.append(feat.cpu())
    return torch.cat(all_features, dim=0)   # [N, 256]

def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path='best_model.pth'):
    best_test_acc = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler.step()
        train_acc, _, _ = test(model, train_loader, device)
        test_acc, test_trues, test_preds = test(model, test_loader, device)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break
    model.load_state_dict(best_model_wts)
    return best_test_acc, best_model_wts

def test(model, loader, device):
    model.eval()
    correct = 0
    preds, trues = [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='评估'):
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(data.y.cpu().numpy())
            correct += (pred == data.y).sum().item()
    accuracy = correct / len(loader.dataset)
    return accuracy, trues, preds

def detailed_test(model, loader, device, models=None):  # 添加 models 参数
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for data in tqdm(loader, desc='详细评估'):
            data = data.to(device)
            if isinstance(model, CrossAttentionFusion):  # 检查是否为融合模型
                if models is None:
                    raise ValueError("models dictionary required for CrossAttentionFusion evaluation")
                # 提取特征列表
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                features_list = [feat_gat, feat_sage, feat_gin]
                out = model(features_list)
            else:
                out = model(data)  # 普通模型直接处理 DataBatch
            prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            pred = out.argmax(dim=1).cpu().numpy()
            true = data.y.cpu().numpy()
            preds.extend(pred)
            trues.extend(true)
            probs.extend(prob)
    acc = accuracy_score(trues, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(trues, preds, average='binary')
    mcc = matthews_corrcoef(trues, preds)
    auc = roc_auc_score(trues, probs)
    tn, fp, fn, tp = confusion_matrix(trues, preds).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics = {'acc': acc, 'mcc': mcc, 'auc': auc, 'sn': sn, 'sp': sp, 'precision': precision, 'recall': recall, 'f1': f1}
    return metrics

def optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=5):
    def objective(trial):
        hidden_dim = trial.suggest_int('hidden_dim', 64, 512)
        num_layers = trial.suggest_int('num_layers', 2, 6)
        dropout = trial.suggest_float('dropout', 0.1, 0.5)
        lr = trial.suggest_loguniform('lr', 1e-4, 1e-2)

        if model_class == DeepGATModel:
            num_heads = trial.suggest_int('num_heads', 2, 16)
            model = DeepGATModel(
                node_feature_dim=model_params['node_feature_dim'],
                edge_feature_dim=model_params['edge_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_heads=num_heads,
                dropout=dropout,
                num_layers=num_layers
            ).to(device)
        elif model_class == GraphSAGEModel:
            model = GraphSAGEModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)
        elif model_class == GINModel:
            model = GINModel(
                node_feature_dim=model_params['node_feature_dim'],
                hidden_dim=hidden_dim,
                out_dim=model_params['out_dim'],
                num_layers=num_layers,
                dropout=dropout
            ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

        best_acc, _ = train_model(
            model, train_loader, test_loader, criterion, optimizer, scheduler,
            device, num_epochs=50, patience=10, model_save_path=f"best_{model_class.__name__}.pth"
        )
        return best_acc

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=5)
    print(f"总试验次数: {len(study.trials)}")
    print(f"{model_class.__name__} 最佳超参数: {study.best_params}")
    for trial in study.trials:
      print(f"Trial {trial.number}: State={trial.state}, Value={trial.value}")
    return study.best_params

def explain_features(model, test_loader, device, output_folder):
    print("正在进行 ESM-C 特征的 SHAP 分析...")
    esmc_features = np.hstack([train_embeddings, train_logits])
    labels = train_labels
    proxy_model = XGBClassifier()
    proxy_model.fit(esmc_features, labels)
    explainer = shap.Explainer(proxy_model)
    shap_values = explainer(esmc_features)
    shap.summary_plot(shap_values, esmc_features, plot_type="bar", show=False)
    plt.title("ESM-C 特征重要性 (SHAP)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "shap_esmc_features.png"), dpi=600)
    plt.close()
    print("SHAP 分析完成，结果已保存至 shap_esmc_features.png")


    print("正在进行 GNNExplainer 分析...")
    trained_model = models['DeepGATModel']
    explainer = GNNExplainer(trained_model, epochs=200, lr=0.01)
    for sample_idx in range(min(5, len(test_struct_data))):
        data = test_struct_data[sample_idx].to(device)
        node_idx = 0  # 分析第一个节点
        node_feat_mask, edge_mask = explainer.explain_node(node_idx, data.x, data.edge_index, data.edge_attr)
        print(f"样本 {sample_idx+1} | 节点 0 特征重要性（前5个）: {node_feat_mask[:5]} | 边重要性（前5个）: {edge_mask[:5]}")
    print("GNNExplainer 分析完成")

    print("正在进行 t-SNE 可视化...")
    def get_last_layer_features(model, loader, device):
        model.eval()
        features = []
        labels = []
        with torch.no_grad():
            for data in loader:
                data = data.to(device)
                feat = model.get_last_layer_features(data)
                features.append(feat.cpu().numpy())
                labels.append(data.y.cpu().numpy())
        return np.vstack(features), np.hstack(labels)

    features, labels = get_last_layer_features(trained_model, test_loader, device)
    tsne = TSNE(n_components=2, random_state=42)
    features_2d = tsne.fit_transform(features)
    plt.figure(figsize=(8, 6))
    plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels, cmap='coolwarm', alpha=0.6)
    plt.title("t-SNE of Last Layer Features (DeepGATModel)")
    plt.colorbar(label='Class')
    plt.savefig(os.path.join(output_folder, "tsne_last_layer.png"), dpi=600)
    plt.close()
    print("t-SNE 可视化完成，结果已保存至 tsne_last_layer.png")

    return results, models

def train_and_evaluate_models(train_loader, test_loader, device, output_folder):
    model_params = {"node_feature_dim": 1156, "edge_feature_dim": 4, "out_dim": 2}
    best_params = {}

    for model_class in [DeepGATModel, GraphSAGEModel, GINModel]:
        print(f"优化 {model_class.__name__}...")
        best_params[model_class.__name__] = optimize_model(model_class, train_loader, test_loader, device, model_params, n_trials=10)

    models = {
        "DeepGATModel": DeepGATModel(
            node_feature_dim=1156, edge_feature_dim=4, out_dim=2,
            hidden_dim=best_params["DeepGATModel"]["hidden_dim"],
            num_layers=best_params["DeepGATModel"]["num_layers"],
            dropout=best_params["DeepGATModel"]["dropout"],
            num_heads=best_params["DeepGATModel"]["num_heads"]
        ).to(device),
        "GraphSAGEModel": GraphSAGEModel(
            node_feature_dim=1156, out_dim=2,
            hidden_dim=best_params["GraphSAGEModel"]["hidden_dim"],
            num_layers=best_params["GraphSAGEModel"]["num_layers"],
            dropout=best_params["GraphSAGEModel"]["dropout"]
        ).to(device),
        "GINModel": GINModel(
            node_feature_dim=1156, out_dim=2,
            hidden_dim=best_params["GINModel"]["hidden_dim"],
            num_layers=best_params["GINModel"]["num_layers"],
            dropout=best_params["GINModel"]["dropout"]
        ).to(device)
    }

    results = {}
    for name, model in models.items():
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=best_params[name]["lr"], weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        save_path = os.path.join(output_folder, f"best_{name}.pth")
        best_acc, _ = train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=10, model_save_path=save_path)
        metrics = detailed_test(model, test_loader, device)
        results[name] = metrics
        print(f"{name} - Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}")

    print("\n### 三个模型性能对比 ###")
    for name, metrics in results.items():
        print(f"{name}: Acc: {metrics['acc']:.4f}, MCC: {metrics['mcc']:.4f}, AUC: {metrics['auc']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}")

    for name, model in models.items():
        model.load_state_dict(torch.load(os.path.join(output_folder, f"best_{name}.pth")))
        model.eval()

    print("\n===== Computing CKA Similarity Between GNN Models =====")

    gin_feats  = collect_embeddings(models["GINModel"],  test_loader, device)
    gat_feats  = collect_embeddings(models["DeepGATModel"], test_loader, device)
    sage_feats = collect_embeddings(models["GraphSAGEModel"], test_loader, device)

    cka_gin_gat   = linear_cka(gin_feats, gat_feats)
    cka_gin_sage  = linear_cka(gin_feats, sage_feats)
    cka_gat_sage  = linear_cka(gat_feats, sage_feats)

    print(f"CKA(GIN, GAT)  = {cka_gin_gat:.4f}")
    print(f"CKA(GIN, SAGE) = {cka_gin_sage:.4f}")
    print(f"CKA(GAT, SAGE) = {cka_gat_sage:.4f}")

    cka_matrix = np.array([
        [1.0,          cka_gat_sage, cka_gin_gat],
        [cka_gat_sage, 1.0,          cka_gin_sage],
        [cka_gin_gat,  cka_gin_sage, 1.0]
    ])

    np.savetxt(os.path.join(output_folder, "cka_matrix.csv"), cka_matrix, delimiter=",")
    plot_cka_heatmap(cka_matrix, output_folder)

    return results, models

# ### 交叉注意力融合训练
# 在交叉注意力融合训练函数中修改
def train_cross_attention_fusion(models, train_loader, test_loader, device, output_folder, num_epochs=50, patience=10):
    fusion_module = CrossAttentionFusion(feature_dim=256, num_heads=4, dropout=0.1).to(device)
    optimizer_fusion = torch.optim.Adam(fusion_module.parameters(), lr=0.001, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler_fusion = torch.optim.lr_scheduler.StepLR(optimizer_fusion, step_size=10, gamma=0.1)

    print("\n### 训练交叉注意力融合模块 ###")
    best_fusion_acc = 0
    best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        fusion_module.train()
        total_loss = 0
        for data in tqdm(train_loader, desc=f'融合训练 Epoch {epoch}/{num_epochs}'):
            data = data.to(device)
            with torch.no_grad():
                feat_gat = models['DeepGATModel'].get_last_layer_features(data).detach()
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data).detach()
                feat_gin = models['GINModel'].get_last_layer_features(data).detach()
                print(f"feat_gat shape: {feat_gat.shape}, type: {type(feat_gat)}")
                print(f"feat_sage shape: {feat_sage.shape}, type: {type(feat_sage)}")
                print(f"feat_gin shape: {feat_gin.shape}, type: {type(feat_gin)}")
                if isinstance(feat_gat, torch.Tensor) and feat_gat.dim() == 2:  # 确保是 [batch_size, feature_dim]
                    features_list = [feat_gat, feat_sage, feat_gin]
                else:
                    raise ValueError("Expected pure tensors from get_last_layer_features, got unexpected type or shape")
            out = fusion_module(features_list)
            loss = criterion(out, data.y)
            optimizer_fusion.zero_grad()
            loss.backward()
            optimizer_fusion.step()
            total_loss += loss.item() * data.num_graphs
        avg_loss = total_loss / len(train_loader.dataset)
        scheduler_fusion.step()

        fusion_module.eval()
        preds, trues = [], []
        with torch.no_grad():
            for data in test_loader:
                data = data.to(device)
                feat_gat = models['DeepGATModel'].get_last_layer_features(data)
                feat_sage = models['GraphSAGEModel'].get_last_layer_features(data)
                feat_gin = models['GINModel'].get_last_layer_features(data)
                if isinstance(feat_gat, torch.Tensor) and feat_gat.dim() == 2:
                    features_list = [feat_gat, feat_sage, feat_gin]
                else:
                    raise ValueError("Expected pure tensors from get_last_layer_features in test loop")
                out = fusion_module(features_list)
                pred = out.argmax(dim=1)
                preds.extend(pred.cpu().numpy())
                trues.extend(data.y.cpu().numpy())
        test_acc = accuracy_score(trues, preds)
        print(f"Epoch: {epoch:02d}, Loss: {avg_loss:.4f}, Fusion Test Acc: {test_acc:.4f}")

        #每5轮或最后一轮保存注意力热力图
        if (epoch % 5 == 0) or (epoch == num_epochs):
            plot_attention_matrix(fusion_module, output_folder, epoch, mode="cross")
            plot_attention_matrix(fusion_module, output_folder, epoch, mode="self")

        if test_acc > best_fusion_acc:
            best_fusion_acc = test_acc
            best_fusion_wts = copy.deepcopy(fusion_module.state_dict())
            epochs_no_improve = 0
            torch.save(fusion_module.state_dict(), os.path.join(output_folder, "best_cross_attention_fusion.pth"))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：在第 {epoch} 轮训练后，无提升，停止训练。")
                break

    fusion_module.load_state_dict(best_fusion_wts)
    fusion_metrics = detailed_test(fusion_module, test_loader, device, models=models)
    return fusion_module, fusion_metrics

In [ ]:
import time, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
import sys, os, datetime

def analyze_feature_correlations(train_struct_data, train_labels, output_folder):
    all_node_features = np.vstack([data.x.numpy() for data in train_struct_data])
    all_edge_features = np.vstack([data.edge_attr.numpy() for data in train_struct_data if data.edge_attr.size(0) > 0])
    labels = np.hstack([data.y.numpy() for data in train_struct_data])
    node_labels = np.repeat(labels, [data.x.shape[0] for data in train_struct_data])

    node_df = pd.DataFrame(all_node_features[:, :10], columns=[f"node_feat_{i}" for i in range(10)])
    node_df['label'] = node_labels
    corr_matrix = node_df.corr()

    plt.figure(figsize=(15, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title("Node Feature Correlation Heatmap")
    plt.savefig(os.path.join(output_folder, "node_feature_correlation.png"), dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()

    if all_edge_features.size > 0:
        edge_df = pd.DataFrame(all_edge_features, columns=['dir_x', 'dir_y', 'dir_z', 'rotation'])
        corr_matrix = edge_df.corr()

        plt.figure(figsize=(8, 6))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
        plt.title("Edge Feature Correlation Heatmap")
        plt.savefig(os.path.join(output_folder, "edge_feature_correlation.png"), dpi=600, bbox_inches='tight', facecolor='white')
        plt.close()

    print("特征相关性分析完成，结果已保存至输出文件夹")

# 长序列分析
def analyze_long_chain_performance(models, fusion_module, test_struct_data, device, output_folder, length_threshold=50):
    
    print("\n===== 长序列样本性能分析=====")

    long_samples = [d for d in test_struct_data if d.x.shape[0] > length_threshold]
    if not long_samples:
        print(f" 未找到节点数 > {length_threshold} 的样本。")
        return

    long_loader = DataLoader(ProteinDataset(long_samples), batch_size=1, shuffle=False)
    lengths, times = [], []

    fusion_module.eval()
    torch.cuda.empty_cache()
    with torch.no_grad():
        for data in long_loader:
            data = data.to(device)
            start = time.time()
            feats = [m.get_last_layer_features(data) for m in models.values()]
            _ = fusion_module(feats)
            torch.cuda.synchronize()
            end = time.time()
            lengths.append(data.x.shape[0])
            times.append(end - start)

    df = pd.DataFrame({"length": lengths, "inference_time (s)": times})
    df.to_csv(os.path.join(output_folder, "test_long_chain_performance_metrics.csv"), index=False)
    plt.scatter(lengths, times, c="steelblue")
    plt.xlabel("Sequence Length (Residues)")
    plt.ylabel("Inference Time (s)")
    plt.title("Long-chain Peptide Inference Time Scaling")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "test_long_chain_scaling.png"), dpi=600)
    plt.close()
    print(" 长序列性能分析完成。")

def analyze_test_performance(models, fusion_module, test_struct_data, test_loader, device, output_folder):
    import time, psutil, matplotlib.pyplot as plt, seaborn as sns
    from sklearn.metrics import confusion_matrix

    print("\n===== 测试集性能分析 =====")
    torch.cuda.empty_cache()

    fusion_module.eval()
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            feats = [m.get_last_layer_features(data) for m in models.values()]
            _ = fusion_module(feats)
    torch.cuda.synchronize()
    total_time = time.time() - start

    avg_time_per_batch = total_time / len(test_loader)
    avg_time_per_sample = total_time / len(test_struct_data)

    gpu_alloc = torch.cuda.memory_allocated(device) / 1024**2
    gpu_reserved = torch.cuda.memory_reserved(device) / 1024**2
    cpu_mem = psutil.Process().memory_info().rss / 1024**2

    perf = {
        "total_inference_time (s)": total_time,
        "avg_inference_time_per_batch (s)": avg_time_per_batch,
        "avg_inference_time_per_sample (s)": avg_time_per_sample,
        "gpu_memory_allocated (MB)": gpu_alloc,
        "gpu_memory_reserved (MB)": gpu_reserved,
        "cpu_memory (MB)": cpu_mem
    }
    pd.DataFrame([perf]).to_csv(os.path.join(output_folder, "final_test_performance_metrics.csv"), index=False)
    print(" 性能指标已保存。")

    analyze_long_chain_performance(models, fusion_module, test_struct_data, device, output_folder)

    print("\n===== 混淆矩阵分析 =====")
    trues, preds = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            feats = [m.get_last_layer_features(data) for m in models.values()]
            out = fusion_module(feats)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            trues.extend(data.y.cpu().numpy())
    cm = confusion_matrix(trues, preds)
    cm_df = pd.DataFrame(cm, index=["True_0", "True_1"], columns=["Pred_0", "Pred_1"])
    cm_df.to_csv(os.path.join(output_folder, "final_test_confusion_matrix.csv"), index=False)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix (Test Set)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "final_test_confusion_matrix.png"), dpi=600)
    plt.close()
    print(" 混淆矩阵图已保存。")

def cross_validation_on_trainset(train_struct_data, train_labels, device, output_folder, k_folds=5):
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    all_metrics = []

    total_samples = len(train_labels)
    print(f"\n===== 开始 {k_folds}-折交叉验证 =====")
    print(f" 总样本数: {total_samples}")
    print(f" 每折平均样本数: {total_samples // k_folds}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(train_labels)), train_labels)):
        print(f"\n===== Fold {fold+1}/{k_folds} =====")

        train_data = [train_struct_data[i] for i in train_idx]
        val_data = [train_struct_data[i] for i in val_idx]
        train_loader = DataLoader(ProteinDataset(train_data), batch_size=32, shuffle=True)
        val_loader = DataLoader(ProteinDataset(val_data), batch_size=32, shuffle=False)

        y_train, y_val = train_labels[train_idx], train_labels[val_idx]
        num_train, num_val = len(train_idx), len(val_idx)
        num_pos_train = np.sum(y_train)
        num_neg_train = num_train - num_pos_train
        num_pos_val = np.sum(y_val)
        num_neg_val = num_val - num_pos_val

        print(f" 当前折样本划分：")
        print(f"  - 训练集样本数: {num_train}  (正样本 {num_pos_train}, 负样本 {num_neg_train})")
        print(f"  - 验证集样本数: {num_val}  (正样本 {num_pos_val}, 负样本 {num_neg_val})")

        results, models = train_and_evaluate_models(train_loader, val_loader, device, output_folder)
        fusion_module, fusion_metrics = train_cross_attention_fusion(models, train_loader, val_loader, device, output_folder)
        results["CrossAttentionFusion"] = fusion_metrics

        for name, metrics in results.items():
            all_metrics.append({
                "fold": fold+1, "model": name,
                **metrics
            })

    df = pd.DataFrame(all_metrics)
    df_mean = df.groupby("model").mean(numeric_only=True).reset_index()
    df_mean["fold"] = "mean"
    df = pd.concat([df, df_mean], ignore_index=True)
    csv_path = os.path.join(output_folder, "five_fold_cross_validation_results.csv")
    df.to_csv(csv_path, index=False)
    print(f" 五折交叉验证完成，结果保存至 {csv_path}")
    return df

def setup_jupyter_logger(output_folder, log_prefix="train"):
    os.makedirs(output_folder, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = os.path.join(output_folder, f"{log_prefix}_log_{timestamp}.log")

    class JupyterLogger(object):
        def __init__(self, filename):
            self.terminal = sys.__stdout__
            self.log = open(filename, "a", encoding="utf-8")
        def write(self, message):
            self.terminal.write(message)
            self.log.write(message)
            self.log.flush()
        def flush(self):
            self.terminal.flush()
            self.log.flush()

    sys.stdout = JupyterLogger(log_path)
    sys.stderr = sys.stdout

    print("=" * 80)
    print(f" 实验启动时间: {timestamp}")
    print(f" 输出目录: {output_folder}")
    print("=" * 80)

    return log_path


In [ ]:
def infer_deepgat_config(state_dict):
    # convs.0.att_src shape: [1, heads, hidden_dim]
    att_src = state_dict["convs.0.att_src"]
    _, num_heads, hidden_dim = att_src.shape

    # edge_preprocess.0.weight shape: [hidden_dim, edge_feature_dim]
    edge_feature_dim = state_dict["edge_preprocess.0.weight"].shape[1]

    # 计算有几层 convs
    num_layers = len([k for k in state_dict.keys() if k.startswith("convs.")])

    return {
        "hidden_dim": hidden_dim,
        "num_heads": num_heads,
        "edge_feature_dim": edge_feature_dim,
        "num_layers": num_layers
    }


def infer_graphsage_hidden_dim(state_dict):
    # SAGEConv: convs.0.lin_l.weight: [hidden_dim, in_dim]
    return state_dict["convs.0.lin_l.weight"].shape[0]


def infer_graphsage_layers(state_dict):
    return len([k for k in state_dict.keys() if k.startswith("convs.")])


def infer_gin_hidden_dim(state_dict):
    # GINConv MLP first layer: convs.0.nn.0.weight: [hidden_dim, node_dim]
    return state_dict["convs.0.nn.0.weight"].shape[0]


def infer_gin_layers(state_dict):
    return len([k for k in state_dict.keys() if k.startswith("convs.")])


def load_trained_models(model_dir, device):
    models = {}

    # ================= DeepGATModel =================
    gat_state = torch.load(os.path.join(model_dir, "best_DeepGATModel.pth"),
                           map_location=device)

    # 从权重自动推断结构
    att_src = gat_state["convs.0.att_src"]          # [1, heads, hidden_dim]
    _, gat_heads, gat_hidden = att_src.shape
    gat_edge_dim = gat_state["edge_preprocess.0.weight"].shape[1]
    gat_conv_keys = [k for k in gat_state.keys() if k.startswith("convs.")]
    gat_layers = len(gat_conv_keys) // 6           # 每层 6 个参数

    print(f"[DeepGAT 配置] hidden_dim={gat_hidden}, heads={gat_heads}, "
          f"edge_dim={gat_edge_dim}, layers={gat_layers}")

    models["DeepGATModel"] = DeepGATModel(
        node_feature_dim=1156,
        edge_feature_dim=gat_edge_dim,
        hidden_dim=gat_hidden,
        num_heads=gat_heads,
        num_layers=gat_layers,
        out_dim=2
    ).to(device)
    models["DeepGATModel"].load_state_dict(gat_state)
    models["DeepGATModel"].eval()

    # ================= GraphSAGEModel =================
    sage_state = torch.load(os.path.join(model_dir, "best_GraphSAGEModel.pth"),
                            map_location=device)

    sage_hidden = sage_state["convs.0.lin_l.weight"].shape[0]
    sage_conv_keys = [k for k in sage_state.keys() if k.startswith("convs.")]
    sage_layers = len(sage_conv_keys) // 3        # 每层 3 个参数

    print(f"[GraphSAGE 配置] hidden_dim={sage_hidden}, layers={sage_layers}")

    models["GraphSAGEModel"] = GraphSAGEModel(
        node_feature_dim=1156,
        hidden_dim=sage_hidden,
        num_layers=sage_layers,
        dropout=0.5,
        out_dim=2
    ).to(device)
    models["GraphSAGEModel"].load_state_dict(sage_state)
    models["GraphSAGEModel"].eval()

    # ================= GINModel =================
    gin_state = torch.load(os.path.join(model_dir, "best_GINModel.pth"),
                           map_location=device)

    gin_hidden = gin_state["convs.0.nn.0.weight"].shape[0]
    gin_conv_keys = [k for k in gin_state.keys() if k.startswith("convs.")]
    gin_layers = len(gin_conv_keys) // 5          # 每层 5 个参数

    print(f"[GIN 配置] hidden_dim={gin_hidden}, layers={gin_layers}")

    models["GINModel"] = GINModel(
        node_feature_dim=1156,
        hidden_dim=gin_hidden,
        num_layers=gin_layers,
        dropout=0.5,
        out_dim=2
    ).to(device)
    models["GINModel"].load_state_dict(gin_state)
    models["GINModel"].eval()

    # ================= CrossAttentionFusion =================
    fusion_state = torch.load(os.path.join(model_dir, "best_cross_attention_fusion.pth"),
                              map_location=device)

    #  融合输入维度 = 三个分支 get_last_layer_features 输出维度 = 256
    fusion_feature_dim = 256

    print(f"[Fusion 配置] feature_dim={fusion_feature_dim}")

    fusion_module = CrossAttentionFusion(
        feature_dim=fusion_feature_dim,
        num_heads=4,
        dropout=0.1
    ).to(device)
    fusion_module.load_state_dict(fusion_state)
    fusion_module.eval()

    return models, fusion_module


def evaluate_on_testset(train_struct_data, test_struct_data, device, output_folder):
    print("\n==== 正在加载已经训练好的模型权重 ====")
  
    model_weight_dir = "esmc_struct_aggregated_9A_CrossAttention"
 
    test_loader = DataLoader(
        ProteinDataset(test_struct_data),
        batch_size=32, shuffle=False
    )
 
    models, fusion_module = load_trained_models(model_weight_dir, device)
 
    results = {}
    for name, model in models.items():
        print(f"\n--- 测试 {name} ---")
        metrics = detailed_test(model, test_loader, device)
        results[name] = metrics
        print(metrics)

    print("\n--- 测试 CrossAttentionFusion ---")
    fusion_metrics = detailed_test(fusion_module, test_loader, device, models=models)
    print(fusion_metrics)
 
    pd.DataFrame(results).to_csv(
        os.path.join(output_folder, "test_results_all_models.csv")
    )
    pd.DataFrame([fusion_metrics]).to_csv(
        os.path.join(output_folder, "test_results_fusion.csv")
    )

    return results, fusion_metrics


In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    log_path = setup_jupyter_logger(aggregated_output_folder, log_prefix="AFP_test")
    print(f" 日志记录开始，输出文件：{log_path}")
 
    test_results, fusion_metrics = evaluate_on_testset(
        train_struct_data,
        test_struct_data,
        device,
        aggregated_output_folder
    )

    print("\n===== 全部测试完成 =====")
    print("三模型结果：", test_results)
    print("融合模型结果：", fusion_metrics)
